# MobiHeal — final paper reproducibility notebook

Executable companion for **“MobiHeal: Post-Quantum Authenticated Handover with One-Hop Recovery from Past-RSU Compromise for the Internet of Vehicles.”**

This version is designed for **paper measurements**, not merely protocol demonstration. In paper mode it requires real standardized **ML-KEM-768** and **ML-DSA-65**, optimized **KMAC256**, durable state transitions, multi-seed statistics, raw timing samples, controlled packet-loss/mobility simulation, measured concurrency, measured admission-gating cost, one-hop compromise-boundary checks, and reproducibility metadata.

### Publication rules enforced by this notebook

1. `MOBIHEAL_PAPER_MODE=1` is the default. Paper mode **fails closed** if a real PQ backend or optimized KMAC implementation is unavailable. It never substitutes emulator timings.
2. The online handover benchmark excludes ML-KEM pool generation/root signing and verifies the ML-DSA pool root before timing, matching the manuscript's separation of offline and online paths.
3. The RSU online timing includes the two crash-consistent SQLite/WAL `synchronous=FULL` commits used by this reference implementation.
4. All randomized evaluation uses multiple independent experiment seeds. Raw samples and seed-level summaries are exported separately; confidence intervals are computed over seed means.
5. Baseline performance values are never invented. A common-platform baseline table remains empty until measured implementations are supplied.
6. Network/mobility outputs are explicitly labeled **controlled protocol-level simulation**, not wireless-testbed measurements.
7. Tamarin results are imported/run only if a `.spthy` model and `tamarin-prover` are actually available.

### Important implementation refinement discovered during paper alignment

The manuscript says the durable offer tuple contains `H(M0)` and cached `M1`, while later stating that an exact retransmitted `M2` can be processed after an RSU crash. Since `TH2` contains `enc(M0)`, `H(M0)` alone is insufficient to reconstruct `TH2` after a crash. This implementation therefore persists the **exact public M0 in addition to H(M0)**. That is the minimum recovery metadata needed to realize the manuscript's stated crash-recovery behavior; `results/manuscript_alignment_notes.csv` records this point so the manuscript can be updated consistently.


In [ ]:
# Optional environment bootstrap. Set MOBIHEAL_AUTO_INSTALL=0 on managed/offline systems.
import importlib.util, os, subprocess, sys

PAPER_MODE = os.environ.get("MOBIHEAL_PAPER_MODE", "1") == "1"
AUTO_INSTALL = os.environ.get("MOBIHEAL_AUTO_INSTALL", "1") == "1"
PQCRYPTO_VERSION = os.environ.get("MOBIHEAL_PQCRYPTO_VERSION", "1.0.0")

needed = []
if importlib.util.find_spec("pqcrypto") is None:
    needed.append(f"pqcrypto=={PQCRYPTO_VERSION}")
if importlib.util.find_spec("Crypto") is None:
    needed.append("pycryptodome==3.23.0")
if importlib.util.find_spec("psutil") is None:
    needed.append("psutil")

if needed and AUTO_INSTALL:
    print("Installing paper dependencies:", needed)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needed])
    except Exception as exc:
        # pqcrypto 1.0.0 wheels require a relatively recent libc on Linux.
        # The legacy 0.4.0 package is still supported by the adapter below.
        if any(x.startswith("pqcrypto==1.0.0") for x in needed):
            print("pqcrypto 1.0.0 installation failed; trying pqcrypto==0.4.0 for compatibility.")
            fallback = ["pqcrypto==0.4.0" if x.startswith("pqcrypto==") else x for x in needed]
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *fallback])
        else:
            raise

if PAPER_MODE and importlib.util.find_spec("pqcrypto") is None:
    raise RuntimeError("Paper mode requires pqcrypto with ML-KEM-768 and ML-DSA-65. Install pqcrypto and rerun.")
if PAPER_MODE and importlib.util.find_spec("Crypto") is None:
    raise RuntimeError("Paper mode requires PyCryptodome KMAC256. Install pycryptodome and rerun.")


Installing paper dependencies: ['pqcrypto==1.0.0', 'pycryptodome==3.23.0']


In [ ]:
from __future__ import annotations

import concurrent.futures
import dataclasses
import hashlib
import hmac
import importlib.metadata
import json
import math
import os
import platform
import random
import secrets
import shutil
import socket
import sqlite3
import statistics
import subprocess
import sys
import tempfile
import threading
import time
import zipfile
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import psutil
except Exception:
    psutil = None

ROOT = Path.cwd()
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"
CLEAN_RESULTS = os.environ.get("MOBIHEAL_CLEAN_RESULTS", "1") == "1"
if CLEAN_RESULTS and RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PLATFORM_LABEL = os.environ.get("MOBIHEAL_PLATFORM_LABEL", f"{platform.machine()}-{socket.gethostname()}")
RUN_STARTED_UTC = pd.Timestamp.utcnow().isoformat()

# Multi-seed paper configuration. Override with MOBIHEAL_SEEDS=2601,2602,... if needed.
def parse_seeds() -> List[int]:
    raw = os.environ.get("MOBIHEAL_SEEDS")
    if raw:
        vals = [int(x.strip()) for x in raw.split(",") if x.strip()]
        if len(vals) < 2:
            raise ValueError("Use at least two independent seeds; ten are recommended for the paper.")
        return vals
    return list(range(2601, 2611)) if PAPER_MODE else [2601, 2602]

SEEDS = parse_seeds()
BOOTSTRAP_REPS = 5000 if PAPER_MODE else 500
HANDOVER_WARMUPS = 10 if PAPER_MODE else 2
HANDOVER_TRIALS_PER_SEED = 100 if PAPER_MODE else 5
PRIMITIVE_WARMUPS = 20 if PAPER_MODE else 2
PRIMITIVE_REPS = {
    "kem_keygen": 200 if PAPER_MODE else 5,
    "kem_encaps": 500 if PAPER_MODE else 10,
    "kem_decaps": 500 if PAPER_MODE else 10,
    "sig_sign": 200 if PAPER_MODE else 5,
    "sig_verify": 500 if PAPER_MODE else 10,
    "kmac": 1000 if PAPER_MODE else 20,
    "sha3": 1000 if PAPER_MODE else 20,
    "merkle_verify": 1000 if PAPER_MODE else 20,
}
POOL_BENCH_SIZES = [64, 256, 1024] if PAPER_MODE else [8, 16]
ACCOUNTING_POOL_SIZES = [64, 256, 1024, 4096]
NETWORK_SESSIONS_PER_CELL = 2000 if PAPER_MODE else 100
NETWORK_LOSS_RATES = [0.00, 0.01, 0.03, 0.05, 0.10, 0.20]
ONE_WAY_DELAYS_MS = [1.0, 5.0, 10.0, 20.0]
VEHICLE_SPEEDS_KMH = [30, 60, 90, 120]
HANDOVER_ZONE_M = [10.0, 25.0, 50.0]
LINK_RATE_MBPS = float(os.environ.get("MOBIHEAL_LINK_RATE_MBPS", "6.0"))
MAX_STAGE_ATTEMPTS = int(os.environ.get("MOBIHEAL_MAX_STAGE_ATTEMPTS", "4"))
RTO_MS = float(os.environ.get("MOBIHEAL_RTO_MS", "50.0"))
CONCURRENCY_LEVELS = [1, 8, 32, 64, 128] if PAPER_MODE else [1, 4]
CONCURRENCY_SESSIONS = 128 if PAPER_MODE else 8
FLOOD_ATTEMPTS = 20000 if PAPER_MODE else 200
CANDIDATE_RSU_COUNTS = [1, 2, 4, 8]
VERIFIER_WINDOWS_S = [1, 5, 10, 30]
PREPOSITION_ARRIVAL_RATE_HZ = float(os.environ.get("MOBIHEAL_PREPOSITION_ARRIVAL_RATE_HZ", "100"))
DEFAULT_POOL_B = 1024

# FIPS 203 / FIPS 204 parameter sizes used by the manuscript.
RID_BYTES = 8
U64_BYTES = 8
NONCE_BYTES = 16
PID_BYTES = 16
TAG_BYTES = 32
SHA3_384_BYTES = 48
MLKEM768_PK_BYTES = 1184
MLKEM768_SK_BYTES = 2400
MLKEM768_CT_BYTES = 1088
MLKEM768_SS_BYTES = 32
MLDSA65_PK_BYTES = 1952
MLDSA65_SK_BYTES = 4032
MLDSA65_SIG_BYTES = 3309

print("MobiHeal final paper configuration")
print("  PAPER_MODE:", PAPER_MODE)
print("  PLATFORM_LABEL:", PLATFORM_LABEL)
print("  SEEDS:", SEEDS)
print("  results directory:", RESULTS_DIR)


MobiHeal final paper configuration
  PAPER_MODE: True
  PLATFORM_LABEL: x86_64-40d79c94c103
  SEEDS: [2601, 2602, 2603, 2604, 2605, 2606, 2607, 2608, 2609, 2610]
  results directory: /content/results


## 1. SHA3-384, KMAC256, domain separation, and deterministic reference cross-check

The manuscript uses SHA3-384 and KMAC256 with mandatory domain-separation strings. Paper mode uses PyCryptodome's optimized KMAC256. A compact dependency-free SP 800-185 implementation is retained only as an executable cross-check and development fallback; paper timing is blocked if optimized KMAC is unavailable.


In [ ]:
# Minimal SP 800-185 KMAC256 reference implementation used only for cross-checks/dev fallback.
_MASK64 = (1 << 64) - 1
_KECCAK_RC = [
    0x0000000000000001,0x0000000000008082,0x800000000000808A,0x8000000080008000,
    0x000000000000808B,0x0000000080000001,0x8000000080008081,0x8000000000008009,
    0x000000000000008A,0x0000000000000088,0x0000000080008009,0x000000008000000A,
    0x000000008000808B,0x800000000000008B,0x8000000000008089,0x8000000000008003,
    0x8000000000008002,0x8000000000000080,0x000000000000800A,0x800000008000000A,
    0x8000000080008081,0x8000000000008080,0x0000000080000001,0x8000000080008008,
]
_KECCAK_ROT = [
    [0,36,3,41,18], [1,44,10,45,2], [62,6,43,15,61],
    [28,55,25,21,56], [27,20,39,8,14],
]

def _rol64(x:int,n:int)->int:
    n%=64
    return ((x<<n)&_MASK64) | (x>>(64-n) if n else x)

def _keccak_f1600(st:List[int])->None:
    for rc in _KECCAK_RC:
        c=[st[x]^st[x+5]^st[x+10]^st[x+15]^st[x+20] for x in range(5)]
        d=[c[(x-1)%5]^_rol64(c[(x+1)%5],1) for x in range(5)]
        for x in range(5):
            for y in range(5): st[x+5*y]^=d[x]
        b=[0]*25
        for x in range(5):
            for y in range(5): b[y+5*((2*x+3*y)%5)] = _rol64(st[x+5*y],_KECCAK_ROT[x][y])
        for x in range(5):
            for y in range(5): st[x+5*y]=b[x+5*y]^((~b[(x+1)%5+5*y])&b[(x+2)%5+5*y])
        st[0]^=rc

def _keccak_sponge(data:bytes,out_len:int,rate:int,suffix:int)->bytes:
    st=[0]*25; padded=bytearray(data); padded.append(suffix)
    while len(padded)%rate != rate-1: padded.append(0)
    padded.append(0x80)
    for off in range(0,len(padded),rate):
        block=padded[off:off+rate]
        for i,b in enumerate(block): st[i//8]^=b<<(8*(i%8))
        _keccak_f1600(st)
    out=bytearray()
    while len(out)<out_len:
        take=min(rate,out_len-len(out))
        for i in range(take): out.append((st[i//8]>>(8*(i%8)))&0xff)
        if len(out)<out_len: _keccak_f1600(st)
    return bytes(out)

def _left_encode(x:int)->bytes:
    n=1
    while x >= 1<<(8*n): n+=1
    return bytes([n])+x.to_bytes(n,"big")

def _right_encode(x:int)->bytes:
    n=1
    while x >= 1<<(8*n): n+=1
    return x.to_bytes(n,"big")+bytes([n])

def _encode_string(s:bytes)->bytes: return _left_encode(8*len(s))+s

def _bytepad(x:bytes,w:int)->bytes:
    z=_left_encode(w)+x
    return z+b"\x00"*((-len(z))%w)

def cshake256_ref(x:bytes,out_len:int,function_name:bytes=b"",customization:bytes=b"")->bytes:
    if not function_name and not customization:
        return _keccak_sponge(x,out_len,136,0x1f)
    prefix=_bytepad(_encode_string(function_name)+_encode_string(customization),136)
    return _keccak_sponge(prefix+x,out_len,136,0x04)

def kmac256_ref(key:bytes,data:bytes,out_len:int=32,customization:bytes=b"")->bytes:
    new_x=_bytepad(_encode_string(key),136)+data+_right_encode(8*out_len)
    return cshake256_ref(new_x,out_len,b"KMAC",customization)

try:
    from Crypto.Hash import KMAC256 as _KMAC256_ACCEL
except Exception:
    _KMAC256_ACCEL = None

if PAPER_MODE and _KMAC256_ACCEL is None:
    raise RuntimeError("Paper mode requires PyCryptodome KMAC256; emulator/reference KMAC timings are not publishable.")

def kmac256(key:bytes,data:bytes,out_len:int=32,customization:bytes=b"")->bytes:
    if _KMAC256_ACCEL is not None:
        return _KMAC256_ACCEL.new(key=key,data=data,mac_len=out_len,custom=customization).digest()
    return kmac256_ref(key,data,out_len,customization)

def H(data:bytes)->bytes:
    return hashlib.sha3_384(data).digest()

LABELS = {
    "AUTH": b"MOBIHEAL-AUTH", "PID": b"MOBIHEAL-PID",
    "ADMIT_CTX": b"MOBIHEAL-ADMIT-CTX", "ADMIT": b"MOBIHEAL-ADMIT",
    "KEM": b"MOBIHEAL-KEM", "TH2": b"MOBIHEAL-TH2",
    "PREAUTH": b"MOBIHEAL-PREAUTH", "TH": b"MOBIHEAL-TH",
    "AUTH_MIX": b"MOBIHEAL-AUTH-MIX", "KEM_MIX": b"MOBIHEAL-KEM-MIX",
    "CONF_V": b"MOBIHEAL-CONF-V", "FIN_V": b"MOBIHEAL-FIN-V",
    "CONF_R": b"MOBIHEAL-CONF-R", "FIN_R": b"MOBIHEAL-FIN-R",
    "APP": b"MOBIHEAL-APP", "SID": b"MOBIHEAL-SID",
}

# Cross-check optimized KMAC against the independent SP 800-185 reference.
_k = bytes(range(32)); _x = b"MobiHeal-KMAC-cross-check"; _s = LABELS["AUTH"]
assert hmac.compare_digest(kmac256_ref(_k,_x,32,_s), kmac256(_k,_x,32,_s))
print("KMAC backend:", "PyCryptodome" if _KMAC256_ACCEL is not None else "pure-Python reference (DEV ONLY)")


KMAC backend: PyCryptodome


## 2. Canonical encoding, exact radio serialization, MIAM derivation, and Merkle authentication

The protocol transcript uses an injective length-delimited canonical encoding. Radio-byte accounting is kept separate: `wire()` methods serialize the fixed fields exactly so the M0/M1/M2/M3 byte totals match the protocol field sizes rather than Python framing overhead.


In [ ]:
def u64(x:int)->bytes:
    if int(x)<0 or int(x)>=1<<64: raise ValueError("uint64 required")
    return int(x).to_bytes(8,"big")

def _canon_atom(x:Any)->Tuple[bytes,bytes]:
    if isinstance(x,bytes): return b"B",x
    if isinstance(x,bytearray): return b"B",bytes(x)
    if isinstance(x,str): return b"S",x.encode()
    if isinstance(x,(int,np.integer)): return b"I",u64(int(x))
    if isinstance(x,(tuple,list)):
        payload=enc(*x); return b"L",payload
    if hasattr(x,"canonical"):
        return b"M",x.canonical()
    raise TypeError(type(x))

def enc(*items:Any)->bytes:
    out=bytearray()
    for item in items:
        typ,payload=_canon_atom(item)
        out += typ + len(payload).to_bytes(4,"big") + payload
    return bytes(out)

def xor_bytes(a:bytes,b:bytes)->bytes:
    if len(a)!=len(b): raise ValueError("XOR length mismatch")
    return bytes(x^y for x,y in zip(a,b))

def derive_miam(S_i:bytes,RID_k:bytes,w:int,c:int)->Tuple[bytes,bytes]:
    if len(S_i)!=32: raise ValueError("S_i must be 256 bits")
    if len(RID_k)!=RID_BYTES: raise ValueError("RID must be 8 bytes in this artifact")
    X = enc(RID_k,w,c)
    A = kmac256(S_i,X,32,LABELS["AUTH"])
    U_pid = kmac256(S_i,X,32,LABELS["PID"])
    return A,U_pid[:PID_BYTES]

def merkle_leaf(RID:bytes,e:int,j:int,epk:bytes)->bytes:
    return H(LABELS["KEM"] + RID + u64(e) + u64(j) + epk)

def merkle_build(leaves:Sequence[bytes])->Tuple[bytes,List[List[bytes]]]:
    if not leaves: raise ValueError("empty Merkle tree")
    level=list(leaves); levels=[level]
    while len(level)>1:
        if len(level)%2: level=level+[level[-1]]
        nxt=[H(level[i]+level[i+1]) for i in range(0,len(level),2)]
        levels.append(nxt); level=nxt
    return level[0],levels

def merkle_path(levels:List[List[bytes]],index0:int)->Tuple[bytes,...]:
    idx=index0; out=[]
    for level in levels[:-1]:
        eff=level if len(level)%2==0 else level+[level[-1]]
        out.append(eff[idx^1]); idx//=2
    return tuple(out)

def merkle_verify(leaf:bytes,index0:int,path:Sequence[bytes],root:bytes)->bool:
    cur=leaf; idx=index0
    for sib in path:
        cur=H(cur+sib) if idx%2==0 else H(sib+cur)
        idx//=2
    return hmac.compare_digest(cur,root)

def root_commitment_bytes(RID:bytes,e:int,root:bytes,B:int)->bytes:
    return RID + u64(e) + root + u64(B)

@dataclass(frozen=True)
class M0:
    PID:bytes; w:int; N0:bytes; alpha0:bytes
    def canonical(self)->bytes: return enc(self.PID,self.w,self.N0,self.alpha0)
    def wire(self)->bytes: return self.PID+u64(self.w)+self.N0+self.alpha0

@dataclass(frozen=True)
class M1:
    RID:bytes; e:int; w:int; j:int; NR:bytes; epk:bytes; path:Tuple[bytes,...]
    def canonical(self)->bytes: return enc(self.RID,self.e,self.w,self.j,self.NR,self.epk,self.path)
    def wire(self)->bytes: return self.RID+u64(self.e)+u64(self.w)+u64(self.j)+self.NR+self.epk+b"".join(self.path)

@dataclass(frozen=True)
class M2:
    PID:bytes; NV:bytes; ct:bytes; alpha2:bytes; tauV:bytes
    def canonical(self)->bytes: return enc(self.PID,self.NV,self.ct,self.alpha2,self.tauV)
    def wire(self)->bytes: return self.PID+self.NV+self.ct+self.alpha2+self.tauV

@dataclass(frozen=True)
class M3:
    sid:bytes; tauR:bytes
    def canonical(self)->bytes: return enc(self.sid,self.tauR)
    def wire(self)->bytes: return self.sid+self.tauR

def parse_m0_wire(raw:bytes)->M0:
    if len(raw)!=72: raise ValueError("bad M0 length")
    return M0(raw[:16],int.from_bytes(raw[16:24],"big"),raw[24:40],raw[40:72])

def parse_m1_wire(raw:bytes,B:int)->M1:
    depth=math.ceil(math.log2(B)) if B>1 else 0
    expected=RID_BYTES+3*8+NONCE_BYTES+MLKEM768_PK_BYTES+depth*SHA3_384_BYTES
    if len(raw)!=expected: raise ValueError(f"bad M1 length {len(raw)} != {expected}")
    o=0; RID=raw[o:o+8]; o+=8
    e=int.from_bytes(raw[o:o+8],"big"); o+=8
    w=int.from_bytes(raw[o:o+8],"big"); o+=8
    j=int.from_bytes(raw[o:o+8],"big"); o+=8
    NR=raw[o:o+16]; o+=16
    epk=raw[o:o+MLKEM768_PK_BYTES]; o+=MLKEM768_PK_BYTES
    path=tuple(raw[o+i*48:o+(i+1)*48] for i in range(depth))
    return M1(RID,e,w,j,NR,epk,path)

def parse_m2_wire(raw:bytes)->M2:
    if len(raw)!=1184: raise ValueError("bad M2 length")
    return M2(raw[:16],raw[16:32],raw[32:1120],raw[1120:1152],raw[1152:1184])

def parse_m3_wire(raw:bytes)->M3:
    if len(raw)!=80: raise ValueError("bad M3 length")
    return M3(raw[:48],raw[48:80])


## 3. Standards-native ML-KEM-768 / ML-DSA-65 backend

Paper mode uses the `pqcrypto` package's standardized ML-KEM-768 and ML-DSA-65 modules. The adapter supports both the current stable API (`keygen/encaps/decaps`) and older `generate_keypair/encrypt/decrypt` releases. It verifies the exact FIPS object sizes before any experiment. A size-accurate emulator exists only for fast development checks when `MOBIHEAL_PAPER_MODE=0`; emulator output is marked `DEV_ONLY` and publication readiness remains blocked.


In [ ]:
class CryptoBackend:
    name="abstract"; standards_native=False; in_process=False
    def kem_keygen(self): raise NotImplementedError
    def kem_encaps(self,pk): raise NotImplementedError
    def kem_decaps(self,sk,ct): raise NotImplementedError
    def sig_keygen(self): raise NotImplementedError
    def sign(self,sk,msg): raise NotImplementedError
    def verify(self,pk,msg,sig): raise NotImplementedError

class PQCryptoBackend(CryptoBackend):
    standards_native=True; in_process=True
    def __init__(self):
        from pqcrypto.kem import ml_kem_768 as kem
        from pqcrypto.sign import ml_dsa_65 as sig
        self.kem=kem; self.sig=sig
        try: ver=importlib.metadata.version("pqcrypto")
        except Exception: ver="unknown"
        self.name=f"pqcrypto-{ver}-ML-KEM-768+ML-DSA-65"
        self._kg=getattr(kem,"keygen",getattr(kem,"generate_keypair",None))
        self._enc=getattr(kem,"encaps",getattr(kem,"encrypt",None))
        self._dec=getattr(kem,"decaps",getattr(kem,"decrypt",None))
        self._skg=getattr(sig,"keygen",getattr(sig,"generate_keypair",None))
        self._sign=getattr(sig,"sign",None); self._verify=getattr(sig,"verify",None)
        if not all([self._kg,self._enc,self._dec,self._skg,self._sign,self._verify]):
            raise RuntimeError("Unsupported pqcrypto API")
        self._selftest()
    def kem_keygen(self):
        pk,sk=self._kg(); return bytes(pk),bytes(sk)
    def kem_encaps(self,pk):
        ct,ss=self._enc(pk); return bytes(ct),bytes(ss)
    def kem_decaps(self,sk,ct): return bytes(self._dec(sk,ct))
    def sig_keygen(self):
        pk,sk=self._skg(); return bytes(pk),bytes(sk)
    def sign(self,sk,msg): return bytes(self._sign(sk,msg))
    def verify(self,pk,msg,sig):
        try:
            r=self._verify(pk,msg,sig)
            return True if r is None else bool(r)
        except Exception:
            return False
    def _selftest(self):
        pk,sk=self.kem_keygen(); ct,z=self.kem_encaps(pk); z2=self.kem_decaps(sk,ct)
        spk,ssk=self.sig_keygen(); msg=b"MobiHeal-pqcrypto-selftest"; sg=self.sign(ssk,msg)
        sizes=(len(pk),len(sk),len(ct),len(z),len(spk),len(ssk),len(sg))
        expected=(MLKEM768_PK_BYTES,MLKEM768_SK_BYTES,MLKEM768_CT_BYTES,MLKEM768_SS_BYTES,MLDSA65_PK_BYTES,MLDSA65_SK_BYTES,MLDSA65_SIG_BYTES)
        if sizes!=expected:
            raise RuntimeError(f"PQC size mismatch: {sizes} != {expected}")
        if not hmac.compare_digest(z,z2) or not self.verify(spk,msg,sg):
            raise RuntimeError("PQC backend self-test failed")

class EmulatedBackend(CryptoBackend):
    name="SIZE-ACCURATE-LOGIC-EMULATOR-DEV-ONLY"; standards_native=False; in_process=True
    def kem_keygen(self):
        sk=secrets.token_bytes(MLKEM768_SK_BYTES)
        pk=hashlib.shake_256(b"pk"+sk).digest(MLKEM768_PK_BYTES); return pk,sk
    def kem_encaps(self,pk):
        r=secrets.token_bytes(32); tail=hashlib.shake_256(b"ct"+pk+r).digest(MLKEM768_CT_BYTES-32)
        ct=r+tail; z=hashlib.sha3_256(b"z"+pk+ct).digest(); return ct,z
    def kem_decaps(self,sk,ct):
        pk=hashlib.shake_256(b"pk"+sk).digest(MLKEM768_PK_BYTES)
        r=ct[:32]; exp=r+hashlib.shake_256(b"ct"+pk+r).digest(MLKEM768_CT_BYTES-32)
        if not hmac.compare_digest(exp,ct): return hashlib.sha3_256(b"reject"+sk+ct).digest()
        return hashlib.sha3_256(b"z"+pk+ct).digest()
    def sig_keygen(self):
        sk=secrets.token_bytes(MLDSA65_SK_BYTES); pk=hashlib.shake_256(b"spk"+sk).digest(MLDSA65_PK_BYTES); return pk,sk
    def sign(self,sk,msg):
        pk=hashlib.shake_256(b"spk"+sk).digest(MLDSA65_PK_BYTES)
        return hashlib.shake_256(b"sig"+pk+msg).digest(MLDSA65_SIG_BYTES)
    def verify(self,pk,msg,sig): return hmac.compare_digest(sig,hashlib.shake_256(b"sig"+pk+msg).digest(MLDSA65_SIG_BYTES))

try:
    BACKEND=PQCryptoBackend()
except Exception as exc:
    if PAPER_MODE:
        raise RuntimeError(f"Paper mode requires a working standards-native pqcrypto backend: {exc}") from exc
    print("WARNING: pqcrypto unavailable; DEV emulator enabled:",exc)
    BACKEND=EmulatedBackend()

EVIDENCE_NATIVE = "native-measured" if BACKEND.standards_native else "DEV_ONLY_EMULATED"
print("Crypto backend:",BACKEND.name)
print("Standards-native:",BACKEND.standards_native)


Crypto backend: pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65
Standards-native: True


## 4. Crash-consistent state store and the M0→M1→M2→M3 state machine

`DurableStateStore` uses SQLite WAL with `synchronous=FULL`. The offer transaction atomically commits the target record, reserved KEM index, `H(M0)`, **exact M0**, and exact cached M1 before M1 is released. The completion transaction atomically commits completed/spent state, `H(M2)`, cached M3, and deletion of the persistent `esk` before M3 is released.

**Erasure boundary:** SQLite/WAL is used here to measure crash-consistent ordering and durability, not as a claim of forensic secure deletion. A theorem-claiming deployment must place `esk` in storage that provides secure erase or crypto-erasure (for example, an HSM/TEE/secure keystore with an erasable wrapping key). Standard filesystem/SSD/WAL deletion can retain stale bytes.


In [ ]:
class DurableStateStore:
    def __init__(self,path:Path,B:int):
        self.path=Path(path); self.B=int(B)
        self.conn=sqlite3.connect(self.path,check_same_thread=False)
        self.conn.execute("PRAGMA journal_mode=WAL")
        self.conn.execute("PRAGMA synchronous=FULL")
        self.conn.execute("PRAGMA temp_store=MEMORY")
        self.conn.executescript("""
        CREATE TABLE IF NOT EXISTS meta(k TEXT PRIMARY KEY, v TEXT NOT NULL);
        CREATE TABLE IF NOT EXISTS target_record(
            pid BLOB NOT NULL, w INTEGER NOT NULL, A BLOB NOT NULL, c INTEGER NOT NULL,
            state TEXT NOT NULL, j INTEGER, m0h BLOB, m0 BLOB, m2h BLOB, m1 BLOB, m3 BLOB,
            PRIMARY KEY(pid,w));
        CREATE TABLE IF NOT EXISTS kem_index(j INTEGER PRIMARY KEY, state TEXT NOT NULL, esk BLOB);
        """)
        self.conn.execute("INSERT OR REPLACE INTO meta(k,v) VALUES('B',?)",(str(self.B),))
        self.conn.commit(); self.lock=threading.RLock()
    def add_record(self,PID,A,w,c):
        with self.lock,self.conn:
            self.conn.execute("INSERT INTO target_record(pid,w,A,c,state) VALUES(?,?,?,?,?)",(PID,int(w),A,int(c),"unused"))
    def add_kem(self,j,sk):
        with self.lock,self.conn:
            self.conn.execute("INSERT INTO kem_index(j,state,esk) VALUES(?,?,?)",(int(j),"free",sk))
    def commit_offer(self,rec,m0,m1,j)->float:
        t0=time.perf_counter_ns()
        with self.lock,self.conn:
            cur=self.conn.execute("SELECT state FROM target_record WHERE pid=? AND w=?",(rec.PID,int(rec.w))).fetchone()
            kem=self.conn.execute("SELECT state FROM kem_index WHERE j=?",(int(j),)).fetchone()
            if cur is None or cur[0]!="unused" or kem is None or kem[0]!="free": raise RuntimeError("offer precondition failed")
            self.conn.execute("UPDATE target_record SET state='offered',j=?,m0h=?,m0=?,m1=? WHERE pid=? AND w=?",
                              (int(j),H(m0.wire()),m0.wire(),m1.wire(),rec.PID,int(rec.w)))
            self.conn.execute("UPDATE kem_index SET state='reserved' WHERE j=?",(int(j),))
        return (time.perf_counter_ns()-t0)/1e6
    def commit_completion(self,rec,m2,m3,j)->float:
        t0=time.perf_counter_ns()
        with self.lock,self.conn:
            self.conn.execute("UPDATE target_record SET state='completed',m2h=?,m3=? WHERE pid=? AND w=?",
                              (H(m2.wire()),m3.wire(),rec.PID,int(rec.w)))
            self.conn.execute("UPDATE kem_index SET state='spent',esk=NULL WHERE j=?",(int(j),))
        return (time.perf_counter_ns()-t0)/1e6
    def commit_abandon(self,rec,j,new_state="expired")->float:
        t0=time.perf_counter_ns()
        with self.lock,self.conn:
            self.conn.execute("UPDATE target_record SET state=? WHERE pid=? AND w=?",(new_state,rec.PID,int(rec.w)))
            self.conn.execute("UPDATE kem_index SET state='abandoned',esk=NULL WHERE j=?",(int(j),))
        return (time.perf_counter_ns()-t0)/1e6
    def snapshot(self,PID,w)->Dict[str,Any]:
        row=self.conn.execute("SELECT state,j,m0h,m0,m2h,m1,m3,A,c FROM target_record WHERE pid=? AND w=?",(PID,int(w))).fetchone()
        if row is None: return {}
        state,j,m0h,m0,m2h,m1,m3,A,c=row
        krow=self.conn.execute("SELECT state,esk FROM kem_index WHERE j=?",(int(j),)).fetchone() if j is not None else None
        return {"state":state,"j":j,"m0h":m0h,"m0":m0,"m2h":m2h,"m1":m1,"m3":m3,"A":A,"c":c,
                "kem_state":None if krow is None else krow[0],"esk":None if krow is None else krow[1]}
    def close(self): self.conn.close()

@dataclass
class TargetRecord:
    PID:bytes; A:bytes; w:int; c:int; state:str="unused"
    accepted_m0_hash:Optional[bytes]=None; accepted_m2_hash:Optional[bytes]=None
    j:Optional[int]=None; cached_m0:Optional[M0]=None; cached_m1:Optional[M1]=None; cached_m3:Optional[M3]=None

@dataclass
class KEMEntry:
    j:int; pk:bytes; sk:Optional[bytes]; state:str="free"

@dataclass
class HandoverContext:
    A:bytes; PID:bytes; m0:M0
    m1:Optional[M1]=None; m2:Optional[M2]=None; Z:Optional[bytes]=None
    XA:Optional[bytes]=None; XZ:Optional[bytes]=None; PRK:Optional[bytes]=None
    Kconf_V:Optional[bytes]=None; Kconf_R:Optional[bytes]=None; Kapp:Optional[bytes]=None
    TH2:Optional[bytes]=None; TH:Optional[bytes]=None; sid:Optional[bytes]=None

class Vehicle:
    def __init__(self,S_i:bytes,c:int=0):
        self.S_i=S_i; self.c=int(c); self.root_cache={}; self.accepted_sessions={}
    def cache_pool_root(self,RID,e,root,B,sig_pk,sig,backend):
        if not backend.verify(sig_pk,root_commitment_bytes(RID,e,root,B),sig): raise ValueError("ML-DSA root verification failed")
        self.root_cache[(RID,e)]=(root,B)
    def start(self,RID,w)->HandoverContext:
        A,PID=derive_miam(self.S_i,RID,w,self.c); N0=secrets.token_bytes(16)
        H0=H(LABELS["ADMIT_CTX"]+enc(RID,w,PID,N0)); alpha0=kmac256(A,H0,32,LABELS["ADMIT"])
        return HandoverContext(A,PID,M0(PID,w,N0,alpha0))
    def process_m1(self,ctx:HandoverContext,m1:M1,backend:CryptoBackend)->M2:
        cached=self.root_cache.get((m1.RID,m1.e))
        if cached is None: raise ValueError("unverified pool root")
        root,B=cached
        if m1.w!=ctx.m0.w or not (1<=m1.j<=B): raise ValueError("M1 context mismatch")
        if not merkle_verify(merkle_leaf(m1.RID,m1.e,m1.j,m1.epk),m1.j-1,m1.path,root): raise ValueError("bad Merkle path")
        ct,Z=backend.kem_encaps(m1.epk); NV=secrets.token_bytes(16)
        T2=enc(ctx.m0)+enc(m1)+enc(ctx.PID)+enc(NV)+enc(ct); TH2=H(LABELS["TH2"]+T2)
        alpha2=kmac256(ctx.A,TH2,32,LABELS["PREAUTH"])
        XA=kmac256(ctx.A,TH2,32,LABELS["AUTH_MIX"]); XZ=kmac256(Z,TH2,32,LABELS["KEM_MIX"]); PRK=xor_bytes(XA,XZ)
        Kconf_V=kmac256(PRK,TH2,32,LABELS["CONF_V"]); tauV=kmac256(Kconf_V,TH2+alpha2,32,LABELS["FIN_V"])
        m2=M2(ctx.PID,NV,ct,alpha2,tauV)
        TH=H(LABELS["TH"]+enc(ctx.m0)+enc(m1)+enc(m2)); Kconf_R=kmac256(PRK,TH,32,LABELS["CONF_R"]); Kapp=kmac256(PRK,TH,32,LABELS["APP"])
        sid=H(LABELS["SID"]+enc(m1.RID,m1.e,m1.w,m1.j,ctx.PID)+TH)
        ctx.m1=m1; ctx.m2=m2; ctx.Z=Z; ctx.XA=XA; ctx.XZ=XZ; ctx.PRK=PRK; ctx.Kconf_V=Kconf_V; ctx.Kconf_R=Kconf_R; ctx.Kapp=Kapp; ctx.TH2=TH2; ctx.TH=TH; ctx.sid=sid
        return m2
    def process_m3(self,ctx:HandoverContext,m3:M3)->bytes:
        if None in (ctx.TH,ctx.sid,ctx.Kconf_R,ctx.Kapp): raise ValueError("incomplete context")
        if not hmac.compare_digest(ctx.sid,m3.sid): raise ValueError("sid mismatch")
        exp=kmac256(ctx.Kconf_R,ctx.TH+ctx.sid,32,LABELS["FIN_R"])
        if not hmac.compare_digest(exp,m3.tauR): raise ValueError("bad RSU confirmation")
        Kapp=ctx.Kapp; self.accepted_sessions[ctx.sid]=Kapp; self.c+=1
        ctx.Z=ctx.XA=ctx.XZ=ctx.PRK=ctx.Kconf_V=ctx.Kconf_R=ctx.Kapp=None
        return Kapp
    def retire_session(self,sid): self.accepted_sessions.pop(sid,None)

class RSU:
    def __init__(self,RID:bytes,backend:CryptoBackend,epoch:int=1,store:Optional[DurableStateStore]=None,sig_pair=None):
        if len(RID)!=8: raise ValueError("RID must be 8 bytes")
        self.RID=RID; self.backend=backend; self.epoch=int(epoch); self.store=store; self.lock=threading.RLock()
        self.records={}; self.pool=[]; self.free_indices=deque(); self.levels=[]; self.root=None; self.root_sig=None
        self.sig_pk,self.sig_sk = sig_pair if sig_pair is not None else backend.sig_keygen()
        self.session_keys={}; self.decap_calls=0; self.last_metrics={}
    def prepare_pool(self,B:int):
        if B<1: raise ValueError("B>=1")
        pool=[]; leaves=[]
        for j in range(1,B+1):
            pk,sk=self.backend.kem_keygen(); pool.append(KEMEntry(j,pk,sk)); leaves.append(merkle_leaf(self.RID,self.epoch,j,pk))
        root,levels=merkle_build(leaves); sig=self.backend.sign(self.sig_sk,root_commitment_bytes(self.RID,self.epoch,root,B))
        self.pool=pool; self.free_indices=deque(range(B)); self.root=root; self.levels=levels; self.root_sig=sig
        if self.store is not None:
            for e in pool: self.store.add_kem(e.j,e.sk)
    def add_record(self,PID,A,w,c):
        key=(PID,int(w));
        if key in self.records: raise ValueError("duplicate target record")
        rec=TargetRecord(PID,A,int(w),int(c)); self.records[key]=rec
        if self.store is not None: self.store.add_record(PID,A,w,c)
    def handle_m0(self,m0:M0)->Tuple[str,Optional[M1]]:
        t0=time.perf_counter_ns(); commit_ms=0.0
        with self.lock:
            rec=self.records.get((m0.PID,m0.w))
            if rec is None: self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":0.0}; return "REJECT_NO_RECORD",None
            m0h=H(m0.wire())
            if rec.state=="offered":
                st="DUPLICATE_M0" if rec.accepted_m0_hash is not None and hmac.compare_digest(rec.accepted_m0_hash,m0h) else "REJECT_CONFLICT"
                self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":0.0}; return st,rec.cached_m1 if st=="DUPLICATE_M0" else None
            if rec.state in ("completed","expired"):
                self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":0.0}; return "REJECT_TERMINAL",None
            H0=H(LABELS["ADMIT_CTX"]+enc(self.RID,m0.w,m0.PID,m0.N0)); exp=kmac256(rec.A,H0,32,LABELS["ADMIT"])
            if not hmac.compare_digest(exp,m0.alpha0):
                self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":0.0}; return "REJECT_ALPHA0",None
            if not self.free_indices:
                self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":0.0}; return "BUSY",None
            idx=self.free_indices[0]; entry=self.pool[idx]; j=entry.j; NR=secrets.token_bytes(16)
            m1=M1(self.RID,self.epoch,m0.w,j,NR,entry.pk,merkle_path(self.levels,idx))
            if self.store is not None: commit_ms=self.store.commit_offer(rec,m0,m1,j)
            self.free_indices.popleft(); entry.state="reserved"; rec.state="offered"; rec.j=j; rec.accepted_m0_hash=m0h; rec.cached_m0=m0; rec.cached_m1=m1
            self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":commit_ms}
            return "OFFERED",m1
    def _find_active_by_pid(self,PID:bytes)->Optional[TargetRecord]:
        m=[r for r in self.records.values() if hmac.compare_digest(r.PID,PID) and r.state in ("offered","completed")]
        return m[0] if len(m)==1 else None
    def handle_m2(self,m2:M2)->Tuple[str,Optional[M3],Optional[bytes]]:
        t0=time.perf_counter_ns(); commit_ms=0.0; decap_ms=0.0
        with self.lock:
            rec=self._find_active_by_pid(m2.PID)
            if rec is None:
                self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":0.0,"decap_ms":0.0}; return "REJECT_NO_OFFER",None,None
            m2h=H(m2.wire())
            if rec.state=="completed":
                st="DUPLICATE_M2" if rec.accepted_m2_hash is not None and hmac.compare_digest(rec.accepted_m2_hash,m2h) else "REJECT_CONFLICT"
                self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":0.0,"decap_ms":0.0}; return st,rec.cached_m3 if st=="DUPLICATE_M2" else None,None
            if rec.cached_m0 is None or rec.cached_m1 is None or rec.j is None:
                raise RuntimeError("offered state missing recovery material")
            m1=rec.cached_m1; entry=self.pool[rec.j-1]
            T2=enc(rec.cached_m0)+enc(m1)+enc(m2.PID)+enc(m2.NV)+enc(m2.ct); TH2=H(LABELS["TH2"]+T2)
            exp_a2=kmac256(rec.A,TH2,32,LABELS["PREAUTH"])
            if not hmac.compare_digest(exp_a2,m2.alpha2):
                self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":0.0,"decap_ms":0.0}; return "REJECT_ALPHA2",None,None
            td=time.perf_counter_ns(); Z=self.backend.kem_decaps(entry.sk,m2.ct); decap_ms=(time.perf_counter_ns()-td)/1e6; self.decap_calls+=1
            XA=kmac256(rec.A,TH2,32,LABELS["AUTH_MIX"]); XZ=kmac256(Z,TH2,32,LABELS["KEM_MIX"]); PRK=xor_bytes(XA,XZ)
            Kconf_V=kmac256(PRK,TH2,32,LABELS["CONF_V"]); exp_tau=kmac256(Kconf_V,TH2+m2.alpha2,32,LABELS["FIN_V"])
            if not hmac.compare_digest(exp_tau,m2.tauV):
                if self.store is not None: commit_ms=self.store.commit_abandon(rec,rec.j,"expired")
                rec.state="expired"; entry.state="abandoned"; entry.sk=None
                self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":commit_ms,"decap_ms":decap_ms}; return "REJECT_TAUV",None,None
            TH=H(LABELS["TH"]+enc(rec.cached_m0)+enc(m1)+enc(m2)); Kconf_R=kmac256(PRK,TH,32,LABELS["CONF_R"]); Kapp=kmac256(PRK,TH,32,LABELS["APP"])
            sid=H(LABELS["SID"]+enc(self.RID,self.epoch,m1.w,m1.j,m2.PID)+TH); tauR=kmac256(Kconf_R,TH+sid,32,LABELS["FIN_R"]); m3=M3(sid,tauR)
            if self.store is not None: commit_ms=self.store.commit_completion(rec,m2,m3,rec.j)
            rec.state="completed"; rec.accepted_m2_hash=m2h; rec.cached_m3=m3; entry.state="spent"; entry.sk=None; self.session_keys[sid]=Kapp
            # Z/XA/XZ/PRK/Kconf values are local variables only and become unreachable on return.
            self.last_metrics={"total_ms":(time.perf_counter_ns()-t0)/1e6,"commit_ms":commit_ms,"decap_ms":decap_ms}
            return "COMPLETED",m3,Kapp
    def expire_offer(self,PID,w)->bool:
        with self.lock:
            rec=self.records.get((PID,int(w)))
            if rec is None or rec.state!="offered" or rec.j is None: return False
            entry=self.pool[rec.j-1]
            if self.store is not None: self.store.commit_abandon(rec,rec.j,"expired")
            rec.state="expired"; entry.state="abandoned"; entry.sk=None
            return True
    def retire_session(self,sid): self.session_keys.pop(sid,None)

def provision(vehicle:Vehicle,rsu:RSU,w:int):
    # Experimental harness emulates MSC prepositioning under the manuscript's protected control-plane assumption.
    A,PID=derive_miam(vehicle.S_i,rsu.RID,w,vehicle.c); rsu.add_record(PID,A,w,vehicle.c); return A,PID


## 5. Protocol, replay, erasure, crash-recovery, and cross-target self-tests

These tests map directly to the manuscript's state-machine and Appendix-B implementation checklist. They are implementation invariants, not substitutes for the AKE/OPMH proofs or Tamarin.


In [ ]:
def run_protocol_selftests(backend:CryptoBackend)->pd.DataFrame:
    rows=[]
    def rec(name,ok,detail=""): rows.append({"test":name,"passed":bool(ok),"detail":detail})
    with tempfile.TemporaryDirectory() as td:
        db=Path(td)/"state.sqlite"; B=8; store=DurableStateStore(db,B)
        RID=b"RSU-0001"; S=hashlib.sha3_256(b"selftest-root").digest(); v=Vehicle(S); r=RSU(RID,backend,epoch=7,store=store); r.prepare_pool(B)
        v.cache_pool_root(RID,r.epoch,r.root,B,r.sig_pk,r.root_sig,backend)
        provision(v,r,101); ctx=v.start(RID,101); st,m1=r.handle_m0(ctx.m0)
        snap_offer=store.snapshot(ctx.PID,101)
        rec("alpha0_before_reservation",st=="OFFERED" and snap_offer["state"]=="offered" and snap_offer["kem_state"]=="reserved")
        rec("durable_offer_contains_exact_M0_and_M1",snap_offer["m0"]==ctx.m0.wire() and snap_offer["m1"]==m1.wire())
        rec("counter_not_radio_field",all(f.name!="c" for T in (M0,M1,M2,M3) for f in dataclasses.fields(T)))
        m2=v.process_m1(ctx,m1,backend); before_c=v.c; st2,m3,kr=r.handle_m2(m2); snap_done=store.snapshot(ctx.PID,101)
        rec("completion_commit_spent_and_esk_deleted",st2=="COMPLETED" and snap_done["state"]=="completed" and snap_done["kem_state"]=="spent" and snap_done["esk"] is None)
        kv=v.process_m3(ctx,m3); rec("shared_Kapp",hmac.compare_digest(kr,kv)); rec("counter_advances_only_after_M3",before_c==0 and v.c==1)
        rec("domain_separated_application_key_not_confirmation_key",ctx.Kapp is None and len(kv)==32)
        # Duplicate completed M2 returns cached M3, no new decap.
        d0=r.decap_calls; sdup,m3dup,_=r.handle_m2(m2); rec("duplicate_M2_cached_M3_no_decap",sdup=="DUPLICATE_M2" and m3dup==m3 and r.decap_calls==d0)
        # Duplicate M0 while offered.
        provision(v,r,102); c2=v.start(RID,102); sa,m1a=r.handle_m0(c2.m0); reserved=sum(x.state=="reserved" for x in r.pool); sb,m1b=r.handle_m0(c2.m0)
        rec("duplicate_M0_cached_M1_no_extra_key",sa=="OFFERED" and sb=="DUPLICATE_M0" and m1a==m1b and sum(x.state=="reserved" for x in r.pool)==reserved)
        # Invalid alpha2 must not decapsulate.
        good=v.process_m1(c2,m1a,backend); bad=dataclasses.replace(good,alpha2=bytes([good.alpha2[0]^1])+good.alpha2[1:]); d0=r.decap_calls; sbad,_,_=r.handle_m2(bad)
        rec("alpha2_gate_before_decap",sbad=="REJECT_ALPHA2" and r.decap_calls==d0 and r.records[(c2.PID,102)].state=="offered")
        # Tampered tauV after valid alpha2 abandons/erases.
        badtau=dataclasses.replace(good,tauV=bytes([good.tauV[0]^1])+good.tauV[1:]); stau,_,_=r.handle_m2(badtau); rr=r.records[(c2.PID,102)]; ee=r.pool[rr.j-1]
        rec("tauV_failure_abandons_and_erases",stau=="REJECT_TAUV" and rr.state=="expired" and ee.state=="abandoned" and ee.sk is None)
        # Cross-target reuse must fail.
        r2=RSU(b"RSU-0002",backend,epoch=7); r2.prepare_pool(2); Aold,PIDold=derive_miam(S,RID,103,v.c); r2.add_record(PIDold,Aold,103,v.c)
        N0=secrets.token_bytes(16); H0old=H(LABELS["ADMIT_CTX"]+enc(RID,103,PIDold,N0)); fake=M0(PIDold,103,N0,kmac256(Aold,H0old,32,LABELS["ADMIT"])); scross,_=r2.handle_m0(fake)
        rec("cross_target_MIAM_replay_rejected",scross=="REJECT_ALPHA0")
        # Merkle tamper rejected at vehicle.
        v2=Vehicle(hashlib.sha3_256(b"m2").digest()); v2.cache_pool_root(r2.RID,r2.epoch,r2.root,2,r2.sig_pk,r2.root_sig,backend); provision(v2,r2,104); c3=v2.start(r2.RID,104); _,mm1=r2.handle_m0(c3.m0)
        tampered=dataclasses.replace(mm1,epk=bytes([mm1.epk[0]^1])+mm1.epk[1:])
        try: v2.process_m1(c3,tampered,backend); ok=False
        except ValueError: ok=True
        rec("tampered_ephemeral_KEM_offer_rejected",ok)
        # Fail-closed timeout.
        provision(v2,r2,105); c4=v2.start(r2.RID,105); _,tm1=r2.handle_m0(c4.m0); ok=r2.expire_offer(c4.PID,105); ent=r2.pool[tm1.j-1]
        rec("offer_timeout_never_recycles_key",ok and ent.state=="abandoned" and ent.sk is None and tm1.j-1 not in r2.free_indices)
        # Live vs retired session store.
        sid=m3.sid; rec("live_Kapp_resident_before_retirement",sid in r.session_keys); r.retire_session(sid); rec("retired_Kapp_erased",sid not in r.session_keys)
        store.close()
    out=pd.DataFrame(rows); return out

protocol_selftests=run_protocol_selftests(BACKEND)
assert protocol_selftests.passed.all(), protocol_selftests.loc[~protocol_selftests.passed]
protocol_selftests


,test,passed,detail
0,alpha0_before_reservation,True,
1,durable_offer_contains_exact_M0_and_M1,True,
2,counter_not_radio_field,True,
3,completion_commit_spent_and_esk_deleted,True,
4,shared_Kapp,True,
5,counter_advances_only_after_M3,True,
6,domain_separated_application_key_not_confirmat...,True,
7,duplicate_M2_cached_M3_no_decap,True,
8,duplicate_M0_cached_M1_no_extra_key,True,
9,alpha2_gate_before_decap,True,


## 6. Host, library, compiler/build, and run metadata

Every paper run records hardware/OS/runtime metadata, the exact Python dependency freeze, and crypto-library version. Use `MOBIHEAL_PLATFORM_LABEL` to distinguish the automotive/edge Linux and ARM runs requested by the manuscript.


In [ ]:
def _read_first(paths):
    for p in paths:
        try:
            x=Path(p).read_text().strip()
            if x: return x
        except Exception: pass
    return ""

def host_metadata()->pd.DataFrame:
    cpu_model=""
    try:
        for line in Path("/proc/cpuinfo").read_text(errors="ignore").splitlines():
            if line.lower().startswith(("model name","hardware")):
                cpu_model=line.split(":",1)[1].strip(); break
    except Exception: cpu_model=platform.processor()
    mem_gib=(psutil.virtual_memory().total/2**30) if psutil else np.nan
    governor=_read_first(["/sys/devices/system/cpu/cpu0/cpufreq/scaling_governor"])
    openssl=""
    if shutil.which("openssl"):
        try: openssl=subprocess.check_output(["openssl","version","-a"],text=True,stderr=subprocess.STDOUT).replace("\n"," | ")
        except Exception as e: openssl=f"ERROR:{e}"
    packages={}
    for p in ["pqcrypto","pycryptodome","numpy","pandas","matplotlib","psutil"]:
        try: packages[p]=importlib.metadata.version(p)
        except Exception: packages[p]="NOT_INSTALLED"
    rows={
        "run_started_utc":RUN_STARTED_UTC,"platform_label":PLATFORM_LABEL,"paper_mode":PAPER_MODE,
        "python":sys.version.replace("\n"," "),"os_platform":platform.platform(),"machine":platform.machine(),
        "cpu_model":cpu_model,"logical_cpu_count":os.cpu_count(),"ram_GiB":mem_gib,"cpu_governor":governor,
        "backend":BACKEND.name,"backend_standards_native":BACKEND.standards_native,"backend_in_process":BACKEND.in_process,
        "kmac_backend":"PyCryptodome" if _KMAC256_ACCEL is not None else "pure-Python DEV ONLY",
        "openssl_build":openssl,"seeds":json.dumps(SEEDS),"packages":json.dumps(packages,sort_keys=True),
    }
    return pd.DataFrame(rows.items(),columns=["key","value"])

ENVIRONMENT=host_metadata()
try:
    PIP_FREEZE=subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True,stderr=subprocess.STDOUT)
except Exception as e:
    PIP_FREEZE=f"ERROR: {e}\n"
ENVIRONMENT


,key,value
0,run_started_utc,2026-09-29T09:35:37.202184+00:00
1,platform_label,x86_64-40d79c94c103
2,paper_mode,True
3,python,"3.13.15 (main, Aug 6 2026, 11:06:22) [GCC 13...."
4,os_platform,Linux-6.6.122+-x86_64-with-glibc2.39
5,machine,x86_64
6,cpu_model,AMD EPYC 7B12
7,logical_cpu_count,2
8,ram_GiB,12.671417
9,cpu_governor,


## 7. Statistics helpers

Raw measurements remain untouched. Summary confidence intervals are bootstrapped over **seed means**, so repeated samples within one run are not treated as independent experimental replicates.


In [ ]:
def bootstrap_mean_ci(seed_means:Sequence[float],reps:int=BOOTSTRAP_REPS,alpha:float=.05,seed:int=99173)->Tuple[float,float]:
    x=np.asarray(seed_means,dtype=float)
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng(seed); idx=rng.integers(0,len(x),size=(reps,len(x))); means=x[idx].mean(axis=1)
    return float(np.quantile(means,alpha/2)),float(np.quantile(means,1-alpha/2))

def summarize_multiseed(df:pd.DataFrame,value_col:str,group_cols:Sequence[str]=())->pd.DataFrame:
    group_cols=list(group_cols); rows=[]
    grouped=[((),df)] if not group_cols else df.groupby(group_cols,dropna=False)
    for key,g in grouped:
        if group_cols and not isinstance(key,tuple): key=(key,)
        raw=g[value_col].dropna().astype(float).to_numpy(); seed_means=g.groupby("seed")[value_col].mean().astype(float).to_numpy()
        lo,hi=bootstrap_mean_ci(seed_means)
        row={c:v for c,v in zip(group_cols,key if group_cols else ())}
        row.update({"metric":value_col,"n_samples":len(raw),"n_seeds":len(seed_means),"mean":float(np.mean(raw)) if len(raw) else np.nan,
                    "sd":float(np.std(raw,ddof=1)) if len(raw)>1 else np.nan,"median":float(np.median(raw)) if len(raw) else np.nan,
                    "p95":float(np.quantile(raw,.95)) if len(raw) else np.nan,"p99":float(np.quantile(raw,.99)) if len(raw) else np.nan,
                    "seed_mean_ci95_low":lo,"seed_mean_ci95_high":hi})
        rows.append(row)
    return pd.DataFrame(rows)

def time_call(fn)->Tuple[Any,float]:
    t0=time.perf_counter_ns(); v=fn(); return v,(time.perf_counter_ns()-t0)/1e6


## 8. Native primitive microbenchmarks

The operations are measured in-process with fixed repetition counts after warm-up. Decapsulation and verification inputs are pre-generated outside their timed regions. The CSV preserves every timing sample.


In [ ]:
def run_primitive_benchmarks(backend:CryptoBackend)->pd.DataFrame:
    rows=[]
    for seed in SEEDS:
        rng=np.random.default_rng(seed)
        # Warm-up core operations.
        pk,sk=backend.kem_keygen(); ct,z=backend.kem_encaps(pk); spk,ssk=backend.sig_keygen(); msg=rng.bytes(96); sg=backend.sign(ssk,msg)
        for _ in range(PRIMITIVE_WARMUPS):
            backend.kem_encaps(pk); backend.kem_decaps(sk,ct); backend.sign(ssk,msg); backend.verify(spk,msg,sg); kmac256(rng.bytes(32),rng.bytes(96),32,LABELS["AUTH"]); H(rng.bytes(2048))
        # keygen
        for i in range(PRIMITIVE_REPS["kem_keygen"]):
            _,ms=time_call(backend.kem_keygen); rows.append((seed,"ML-KEM-768","keygen",i,ms))
        # encaps
        for i in range(PRIMITIVE_REPS["kem_encaps"]):
            _,ms=time_call(lambda: backend.kem_encaps(pk)); rows.append((seed,"ML-KEM-768","encaps",i,ms))
        # decaps with pre-generated ciphertexts
        cts=[backend.kem_encaps(pk)[0] for _ in range(PRIMITIVE_REPS["kem_decaps"])]
        for i,c in enumerate(cts):
            _,ms=time_call(lambda c=c: backend.kem_decaps(sk,c)); rows.append((seed,"ML-KEM-768","decaps",i,ms))
        # sign
        messages=[rng.bytes(96) for _ in range(PRIMITIVE_REPS["sig_sign"])]
        for i,m in enumerate(messages):
            _,ms=time_call(lambda m=m: backend.sign(ssk,m)); rows.append((seed,"ML-DSA-65","sign",i,ms))
        # verify
        vm=[rng.bytes(96) for _ in range(PRIMITIVE_REPS["sig_verify"])]; vs=[backend.sign(ssk,m) for m in vm]
        for i,(m,s) in enumerate(zip(vm,vs)):
            (_,ms)=time_call(lambda m=m,s=s: backend.verify(spk,m,s)); rows.append((seed,"ML-DSA-65","verify",i,ms))
        # KMAC / SHA3
        kk=rng.bytes(32); xd=rng.bytes(96); hd=rng.bytes(2048)
        for i in range(PRIMITIVE_REPS["kmac"]):
            _,ms=time_call(lambda: kmac256(kk,xd,32,LABELS["AUTH_MIX"])); rows.append((seed,"KMAC256","32-byte output",i,ms))
        for i in range(PRIMITIVE_REPS["sha3"]):
            _,ms=time_call(lambda: H(hd)); rows.append((seed,"SHA3-384","2048-byte input",i,ms))
        # Merkle verification depth 10 / B=1024 using synthetic leaves.
        leaves=[H(rng.bytes(64)) for _ in range(1024)]; root,levels=merkle_build(leaves); idx=seed%1024; path=merkle_path(levels,idx)
        for i in range(PRIMITIVE_REPS["merkle_verify"]):
            _,ms=time_call(lambda: merkle_verify(leaves[idx],idx,path,root)); rows.append((seed,"Merkle-SHA3-384","verify-depth-10",i,ms))
    out=pd.DataFrame(rows,columns=["seed","algorithm","operation","trial","time_ms"])
    out["platform_label"]=PLATFORM_LABEL; out["backend"]=BACKEND.name; out["evidence_class"]=EVIDENCE_NATIVE
    return out

primitive_raw=run_primitive_benchmarks(BACKEND)
primitive_summary=pd.concat([summarize_multiseed(g,"time_ms") .assign(algorithm=a,operation=o)
                             for (a,o),g in primitive_raw.groupby(["algorithm","operation"])],ignore_index=True)
primitive_summary["platform_label"]=PLATFORM_LABEL; primitive_summary["backend"]=BACKEND.name; primitive_summary["evidence_class"]=EVIDENCE_NATIVE
primitive_summary


,metric,n_samples,n_seeds,mean,sd,median,p95,p99,seed_mean_ci95_low,seed_mean_ci95_high,algorithm,operation,platform_label,backend,evidence_class
0,time_ms,10000,10,0.025024,0.142476,0.015860,0.033361,0.049491,0.017796,0.036346,KMAC256,32-byte output,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
1,time_ms,2000,10,11.657736,1.745856,10.991359,16.898834,17.972107,11.253814,12.236039,ML-DSA-65,sign,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
2,time_ms,5000,10,0.337595,0.543697,0.225840,0.407248,4.358706,0.232005,0.490128,ML-DSA-65,verify,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
3,time_ms,5000,10,0.133911,0.192039,0.105730,0.177461,0.235511,0.110204,0.166665,ML-KEM-768,decaps,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
4,time_ms,5000,10,0.100623,0.030923,0.083415,0.147831,0.182793,0.086603,0.116166,ML-KEM-768,encaps,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
5,time_ms,2000,10,0.079442,0.025778,0.064720,0.118182,0.175290,0.067934,0.093341,ML-KEM-768,keygen,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
6,time_ms,10000,10,0.016412,0.101753,0.009470,0.019010,0.029922,0.011114,0.023911,Merkle-SHA3-384,verify-depth-10,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
7,time_ms,10000,10,0.009041,0.038237,0.006670,0.012860,0.017291,0.007099,0.011214,SHA3-384,2048-byte input,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured


## 9. Direct online MobiHeal handover benchmark with durable state

Each seed builds one authenticated ephemeral pool offline, prepositions target records, verifies/caches the signed pool root, performs warm-ups, then times only M0→M3. The reported RSU time includes SQLite `FULL` durable offer/completion commits. Pool generation, ML-DSA root signing, and initial root verification are not counted as per-handover online cost.


In [ ]:
def run_timed_handovers(backend:CryptoBackend)->pd.DataFrame:
    rows=[]
    need=HANDOVER_WARMUPS+HANDOVER_TRIALS_PER_SEED
    B=1<<math.ceil(math.log2(max(DEFAULT_POOL_B,need)))
    for seed in SEEDS:
        with tempfile.TemporaryDirectory() as td:
            store=DurableStateStore(Path(td)/"state.sqlite",B)
            r=RSU(b"RSU-PAPR",backend,epoch=seed,store=store); r.prepare_pool(B)
            vehicles=[]
            for i in range(need):
                S=hashlib.sha3_256(f"MobiHeal-paper-{seed}-{i}".encode()).digest(); v=Vehicle(S); w=seed*1_000_000+i
                provision(v,r,w); v.cache_pool_root(r.RID,r.epoch,r.root,B,r.sig_pk,r.root_sig,backend); vehicles.append((v,w))
            for i,(v,w) in enumerate(vehicles):
                t=time.perf_counter_ns(); ctx=v.start(r.RID,w); t1=time.perf_counter_ns()
                s0,m1=r.handle_m0(ctx.m0); t2=time.perf_counter_ns(); m0m=dict(r.last_metrics)
                m2=v.process_m1(ctx,m1,backend); t3=time.perf_counter_ns()
                s2,m3,kr=r.handle_m2(m2); t4=time.perf_counter_ns(); m2m=dict(r.last_metrics)
                kv=v.process_m3(ctx,m3); t5=time.perf_counter_ns()
                if s0!="OFFERED" or s2!="COMPLETED" or not hmac.compare_digest(kr,kv): raise AssertionError("handover failed")
                if i>=HANDOVER_WARMUPS:
                    rows.append({
                        "seed":seed,"trial":i-HANDOVER_WARMUPS,
                        "vehicle_m0_ms":(t1-t)/1e6,"rsu_m0_offer_ms":(t2-t1)/1e6,
                        "vehicle_m1_to_m2_ms":(t3-t2)/1e6,"rsu_m2_to_m3_ms":(t4-t3)/1e6,
                        "vehicle_m3_verify_ms":(t5-t4)/1e6,
                        "offer_commit_ms":m0m.get("commit_ms",0.0),"completion_commit_ms":m2m.get("commit_ms",0.0),
                        "decapsulation_inside_rsu_ms":m2m.get("decap_ms",np.nan),
                        "vehicle_total_ms":((t1-t)+(t3-t2)+(t5-t4))/1e6,
                        "rsu_total_ms":((t2-t1)+(t4-t3))/1e6,
                        "online_local_total_ms":(t5-t)/1e6,
                        "M0_bytes":len(ctx.m0.wire()),"M1_bytes":len(m1.wire()),"M2_bytes":len(m2.wire()),"M3_bytes":len(m3.wire()),
                        "backend":backend.name,"platform_label":PLATFORM_LABEL,"evidence_class":EVIDENCE_NATIVE,
                    })
                r.retire_session(m3.sid); v.retire_session(m3.sid)
            store.close()
    return pd.DataFrame(rows)

handover_raw=run_timed_handovers(BACKEND)
handover_metrics=["vehicle_m0_ms","rsu_m0_offer_ms","vehicle_m1_to_m2_ms","rsu_m2_to_m3_ms","vehicle_m3_verify_ms","offer_commit_ms","completion_commit_ms","vehicle_total_ms","rsu_total_ms","online_local_total_ms"]
handover_summary=pd.concat([summarize_multiseed(handover_raw,m).assign(platform_label=PLATFORM_LABEL,backend=BACKEND.name,evidence_class=EVIDENCE_NATIVE) for m in handover_metrics],ignore_index=True)
handover_summary


,metric,n_samples,n_seeds,mean,sd,median,p95,p99,seed_mean_ci95_low,seed_mean_ci95_high,platform_label,backend,evidence_class
0,vehicle_m0_ms,1000,10,0.148044,0.044956,0.141635,0.200901,0.244801,0.145295,0.150621,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
1,rsu_m0_offer_ms,1000,10,2.829937,0.920530,2.745700,3.779888,5.628120,2.654786,2.990564,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
2,vehicle_m1_to_m2_ms,1000,10,0.639954,0.104924,0.627160,0.780389,1.029366,0.627773,0.651745,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
3,rsu_m2_to_m3_ms,1000,10,3.242577,0.655819,3.171025,4.341742,5.638805,3.105545,3.385910,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
4,vehicle_m3_verify_ms,1000,10,0.121615,0.035489,0.127094,0.173692,0.212128,0.111628,0.129021,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
5,offer_commit_ms,1000,10,2.745172,0.916416,2.657660,3.692089,5.488588,2.571858,2.903970,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
6,completion_commit_ms,1000,10,2.692644,0.642265,2.632480,3.738256,4.753765,2.535747,2.848782,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
7,vehicle_total_ms,1000,10,0.909613,0.127308,0.899276,1.098281,1.290553,0.893613,0.924130,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
8,rsu_total_ms,1000,10,6.072514,1.400166,5.953185,7.737459,10.508992,5.765099,6.377599,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
9,online_local_total_ms,1000,10,6.982127,1.445514,6.871985,8.701327,11.525963,6.668371,7.295605,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured


## 10. Exact M0/M1/M2/M3 communication and storage accounting

The steady-state radio total assumes the ML-DSA-authenticated pool root/certificate is already cached, as allowed by the protocol. Cold-cache root material is reported separately and excludes a deployment-specific certificate-chain size.


In [ ]:
def semantic_sizes(B:int)->Dict[str,int]:
    depth=math.ceil(math.log2(B)) if B>1 else 0
    m0=PID_BYTES+U64_BYTES+NONCE_BYTES+TAG_BYTES
    m1=RID_BYTES+3*U64_BYTES+NONCE_BYTES+MLKEM768_PK_BYTES+depth*SHA3_384_BYTES
    m2=PID_BYTES+NONCE_BYTES+MLKEM768_CT_BYTES+TAG_BYTES+TAG_BYTES
    m3=SHA3_384_BYTES+TAG_BYTES
    return {"B":B,"merkle_depth":depth,"M0_bytes":m0,"M1_bytes":m1,"M2_bytes":m2,"M3_bytes":m3,"TOTAL_bytes":m0+m1+m2+m3,
            "cold_cache_root_pk_sig_bytes_excl_cert":SHA3_384_BYTES+MLDSA65_PK_BYTES+MLDSA65_SIG_BYTES}

message_sizes=pd.DataFrame([semantic_sizes(B) for B in ACCOUNTING_POOL_SIZES])
# Validate the default values using an actual generated handover sample.
assert set(handover_raw[["M0_bytes","M2_bytes","M3_bytes"]].iloc[0])=={72,1184,80}
assert semantic_sizes(DEFAULT_POOL_B)["TOTAL_bytes"]==3048

storage_rows=[]
for B in ACCOUNTING_POOL_SIZES:
    levels=[]; n=B
    while n>1:
        if n%2: n+=1
        n//=2; levels.append(n)
    merkle_internal_nodes=sum(levels)
    raw_pool=B*(MLKEM768_PK_BYTES+MLKEM768_SK_BYTES)
    storage_rows.append({"B":B,"KEM_key_material_bytes":raw_pool,"logical_state_bytes":B,
                         "Merkle_internal_hash_bytes":merkle_internal_nodes*SHA3_384_BYTES,
                         "signed_root_bytes_excl_cert":SHA3_384_BYTES+MLDSA65_SIG_BYTES,
                         "RSU_crypto_storage_bytes":raw_pool+B+merkle_internal_nodes*SHA3_384_BYTES+SHA3_384_BYTES+MLDSA65_SIG_BYTES,
                         "RSU_crypto_storage_MiB":(raw_pool+B+merkle_internal_nodes*SHA3_384_BYTES+SHA3_384_BYTES+MLDSA65_SIG_BYTES)/2**20,
                         "evidence_class":"accounting"})
pool_storage=pd.DataFrame(storage_rows)

# Semantic memory/storage accounting with implementation overhead intentionally separated.
default_m=semantic_sizes(DEFAULT_POOL_B)
memory_storage_accounting=pd.DataFrame([
    {"component":"vehicle_root_secret_plus_counter","bytes":32+8,"scope":"per vehicle","evidence_class":"accounting","note":"protected S_i plus hidden uint64 counter"},
    {"component":"vehicle_verified_pool_cache_minimum","bytes":RID_BYTES+8+8+SHA3_384_BYTES,"scope":"per cached RSU epoch","evidence_class":"accounting","note":"RID,e,B,root only; excludes certificate/public-key cache"},
    {"component":"vehicle_pool_cache_with_MLDSA_pk_and_signature_excl_cert","bytes":RID_BYTES+8+8+SHA3_384_BYTES+MLDSA65_PK_BYTES+MLDSA65_SIG_BYTES,"scope":"per cached RSU epoch","evidence_class":"accounting","note":"certificate chain remains deployment-specific"},
    {"component":"target_record_semantic","bytes":PID_BYTES+32+8+8+1,"scope":"per target/window/counter record","evidence_class":"accounting","note":"PID,A,w,c,logical state; excludes DB/index overhead"},
    {"component":"completed_retransmission_cache_B1024","bytes":48+48+default_m["M1_bytes"]+default_m["M3_bytes"]+8,"scope":"per retained completed session","evidence_class":"accounting","note":"H(M0),H(M2),M1,M3,expiry; excludes DB/index overhead"},
    {"component":"offer_recovery_cache_B1024_with_exact_M0_refinement","bytes":48+default_m["M0_bytes"]+8+16+default_m["M1_bytes"]+8,"scope":"per active offer","evidence_class":"accounting","note":"H(M0),M0,j,NR,M1,toffer; exact M0 is the recovery refinement"},
])
message_sizes, pool_storage, memory_storage_accounting


(      B  merkle_depth  M0_bytes  M1_bytes  M2_bytes  M3_bytes  TOTAL_bytes  \
 0    64             6        72      1520      1184        80         2856   
 1   256             8        72      1616      1184        80         2952   
 2  1024            10        72      1712      1184        80         3048   
 3  4096            12        72      1808      1184        80         3144   
 
    cold_cache_root_pk_sig_bytes_excl_cert  
 0                                    5309  
 1                                    5309  
 2                                    5309  
 3                                    5309  ,
       B  KEM_key_material_bytes  logical_state_bytes  \
 0    64                  229376                   64   
 1   256                  917504                  256   
 2  1024                 3670016                 1024   
 3  4096                14680064                 4096   
 
    Merkle_internal_hash_bytes  signed_root_bytes_excl_cert  \
 0                        3

## 11. Measured offline pool preparation and per-key amortization

This directly measures B independent ML-KEM-768 key generations, Merkle-tree construction, and one ML-DSA-65 signature using a pre-existing RSU identity key. The identity-key generation itself is enrollment/setup and is excluded.


In [ ]:
def benchmark_pool_prep(backend:CryptoBackend)->pd.DataFrame:
    rows=[]
    for seed in SEEDS:
        sig_pair=backend.sig_keygen()  # enrollment/setup; deliberately outside timing
        for B in POOL_BENCH_SIZES:
            RID=b"RSU-POOL"; e=seed
            t0=time.perf_counter_ns(); pool=[]; leaves=[]
            for j in range(1,B+1):
                pk,sk=backend.kem_keygen(); pool.append((pk,sk)); leaves.append(merkle_leaf(RID,e,j,pk))
            t1=time.perf_counter_ns(); root,levels=merkle_build(leaves); t2=time.perf_counter_ns(); sig=backend.sign(sig_pair[1],root_commitment_bytes(RID,e,root,B)); t3=time.perf_counter_ns()
            rows.append({"seed":seed,"B":B,"kem_keygen_total_ms":(t1-t0)/1e6,"merkle_build_ms":(t2-t1)/1e6,"root_sign_ms":(t3-t2)/1e6,
                         "pool_total_ms":(t3-t0)/1e6,"amortized_pool_ms_per_key":(t3-t0)/1e6/B,"root_signature_bytes":len(sig),
                         "platform_label":PLATFORM_LABEL,"backend":backend.name,"evidence_class":EVIDENCE_NATIVE})
    return pd.DataFrame(rows)

pool_prep_raw=benchmark_pool_prep(BACKEND)
pool_prep_summary=pd.concat([summarize_multiseed(g,"pool_total_ms").assign(B=B,measure="pool_total_ms") for B,g in pool_prep_raw.groupby("B")]+[summarize_multiseed(g,"amortized_pool_ms_per_key").assign(B=B,measure="amortized_pool_ms_per_key") for B,g in pool_prep_raw.groupby("B")],ignore_index=True)
pool_prep_summary["platform_label"]=PLATFORM_LABEL; pool_prep_summary["backend"]=BACKEND.name; pool_prep_summary["evidence_class"]=EVIDENCE_NATIVE
pool_prep_summary


,metric,n_samples,n_seeds,mean,sd,median,p95,p99,seed_mean_ci95_low,seed_mean_ci95_high,B,measure,platform_label,backend,evidence_class
0,pool_total_ms,10,10,16.331623,0.435883,16.173925,16.990308,17.206606,16.103331,16.608961,64,pool_total_ms,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
1,pool_total_ms,10,10,30.125908,1.198266,29.675410,32.100868,33.153422,29.622725,30.921001,256,pool_total_ms,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
2,pool_total_ms,10,10,87.307816,3.330610,86.391467,92.965129,95.602755,85.841296,89.446983,1024,pool_total_ms,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
3,amortized_pool_ms_per_key,10,10,0.255182,0.006811,0.252718,0.265474,0.268853,0.251615,0.259515,64,amortized_pool_ms_per_key,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
4,amortized_pool_ms_per_key,10,10,0.117679,0.004681,0.115920,0.125394,0.129506,0.115714,0.120785,256,amortized_pool_ms_per_key,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured
5,amortized_pool_ms_per_key,10,10,0.085262,0.003253,0.084367,0.090786,0.093362,0.083829,0.087351,1024,amortized_pool_ms_per_key,x86_64-40d79c94c103,pqcrypto-1.0.0-ML-KEM-768+ML-DSA-65,native-measured


## 12. Controlled packet-loss / delay / speed / dwell-time evaluation

This is a protocol-level Monte-Carlo experiment parameterized by the **measured online stage times** from this run. It models the exact idempotent recovery structure:

- `M0→M1`: retry the same M0; an existing offer returns cached M1 without allocating another KEM key.
- `M2→M3`: retry the same M2; after completion, cached M3 is returned without a second decapsulation.

The simulator varies independent per-message loss, one-way propagation delay, speed, and handover-zone length. Dwell time is `zone / speed`, rather than the full RSU coverage radius, so mobility can actually constrain retries. Results must be described as controlled simulation unless replaced with SUMO/ns-3/testbed measurements.


In [ ]:
PROFILE={m:float(handover_raw[m].median()) for m in ["vehicle_m0_ms","rsu_m0_offer_ms","vehicle_m1_to_m2_ms","rsu_m2_to_m3_ms","vehicle_m3_verify_ms"]}
SIZE_DEFAULT=semantic_sizes(DEFAULT_POOL_B)

def tx_ms(nbytes:float)->float: return nbytes*8/(LINK_RATE_MBPS*1e6)*1000.0

def simulate_pair(rng,n,p,req_bytes,resp_bytes,processor_ms):
    # Request/response attempts. A failed attempt consumes one RTO; successful attempt uses actual path time.
    req_lost=rng.random((n,MAX_STAGE_ATTEMPTS))<p
    resp_lost=rng.random((n,MAX_STAGE_ATTEMPTS))<p
    success=(~req_lost)&(~resp_lost)
    any_s=success.any(axis=1)
    first=np.where(any_s,success.argmax(axis=1),MAX_STAGE_ATTEMPTS)
    attempts=np.where(any_s,first+1,MAX_STAGE_ATTEMPTS)
    path=tx_ms(req_bytes)+tx_ms(resp_bytes)+2*0.0+processor_ms  # propagation added by caller
    return any_s,first,attempts,req_lost,resp_lost,path

def network_condition(seed:int,loss:float,delay_ms:float,speed_kmh:float,zone_m:float,n:int):
    rng=np.random.default_rng(seed*1000003+int(loss*10000)*101+int(delay_ms*10)*17+int(speed_kmh)*3+int(zone_m))
    a_s,a_first,a_attempts,a_req_loss,a_resp_loss,a_path=simulate_pair(rng,n,loss,SIZE_DEFAULT["M0_bytes"],SIZE_DEFAULT["M1_bytes"],PROFILE["rsu_m0_offer_ms"])
    b_s,b_first,b_attempts,b_req_loss,b_resp_loss,b_path=simulate_pair(rng,n,loss,SIZE_DEFAULT["M2_bytes"],SIZE_DEFAULT["M3_bytes"],PROFILE["rsu_m2_to_m3_ms"])
    # Stage B only starts if stage A succeeded.
    b_s=b_s & a_s
    a_success_path=a_path+2*delay_ms
    b_success_path=b_path+2*delay_ms
    fail_attempt=max(RTO_MS,a_success_path,b_success_path)
    lat=np.full(n,np.nan)
    completed=a_s & b_s
    lat[completed]=PROFILE["vehicle_m0_ms"] + a_first[completed]*fail_attempt + a_success_path + PROFILE["vehicle_m1_to_m2_ms"] + b_first[completed]*fail_attempt + b_success_path + PROFILE["vehicle_m3_verify_ms"]
    dwell_ms=zone_m/(speed_kmh/3.6)*1000.0
    success=completed & (lat<=dwell_ms)
    used_a=np.minimum(a_attempts,MAX_STAGE_ATTEMPTS); used_b=np.where(a_s,np.minimum(b_attempts,MAX_STAGE_ATTEMPTS),0)
    out={"seed":seed,"loss":loss,"one_way_delay_ms":delay_ms,"speed_kmh":speed_kmh,"handover_zone_m":zone_m,"dwell_ms":dwell_ms,"sessions":n,
         "success_rate":float(success.mean()),"protocol_completion_rate":float(completed.mean()),"dwell_failure_rate":float((completed & ~success).mean()),
         "mean_latency_ms":float(np.nanmean(lat[success])) if success.any() else np.nan,"median_latency_ms":float(np.nanmedian(lat[success])) if success.any() else np.nan,
         "p95_latency_ms":float(np.nanquantile(lat[success],.95)) if success.any() else np.nan,"p99_latency_ms":float(np.nanquantile(lat[success],.99)) if success.any() else np.nan,
         "mean_M0_attempts":float(used_a.mean()),"mean_M2_attempts":float(used_b.mean()),
         "M0_retransmission_rate":float(((used_a-1).clip(min=0)).sum()/max(1,used_a.sum())),
         "M2_retransmission_rate":float(((used_b-1).clip(min=0)).sum()/max(1,used_b.sum())),
         "desynchronization_incidence":0.0,
         "evidence_class":"controlled-simulation-from-measured-stage-times","platform_label":PLATFORM_LABEL}
    return out,lat,success

network_rows=[]; canonical_samples=[]
for seed in SEEDS:
    for loss in NETWORK_LOSS_RATES:
        for delay in ONE_WAY_DELAYS_MS:
            for speed in VEHICLE_SPEEDS_KMH:
                for zone in HANDOVER_ZONE_M:
                    row,lat,ok=network_condition(seed,loss,delay,speed,zone,NETWORK_SESSIONS_PER_CELL); network_rows.append(row)
                    if delay==10.0 and speed==90 and zone==25.0:
                        canonical_samples.extend({"seed":seed,"loss":loss,"latency_ms":float(x),"success":bool(s)} for x,s in zip(lat,ok) if not np.isnan(x))
network_seed=pd.DataFrame(network_rows); network_latency_samples=pd.DataFrame(canonical_samples)
network_summary=(network_seed.groupby(["loss","one_way_delay_ms","speed_kmh","handover_zone_m"],as_index=False)
                 .agg(success_rate=("success_rate","mean"),success_rate_sd=("success_rate","std"),protocol_completion_rate=("protocol_completion_rate","mean"),
                      dwell_failure_rate=("dwell_failure_rate","mean"),mean_latency_ms=("mean_latency_ms","mean"),p95_latency_ms=("p95_latency_ms","mean"),
                      p99_latency_ms=("p99_latency_ms","mean"),M0_retransmission_rate=("M0_retransmission_rate","mean"),M2_retransmission_rate=("M2_retransmission_rate","mean"),
                      desynchronization_incidence=("desynchronization_incidence","max")))
network_summary["evidence_class"]="controlled-simulation-from-measured-stage-times"; network_summary["platform_label"]=PLATFORM_LABEL
network_summary.head()


,loss,one_way_delay_ms,speed_kmh,handover_zone_m,success_rate,success_rate_sd,protocol_completion_rate,dwell_failure_rate,mean_latency_ms,p95_latency_ms,p99_latency_ms,M0_retransmission_rate,M2_retransmission_rate,desynchronization_incidence,evidence_class,platform_label
0,0.0,1.0,30,10.0,1.0,0.0,1.0,0.0,14.876615,14.876615,14.876615,0.0,0.0,0.0,controlled-simulation-from-measured-stage-times,x86_64-40d79c94c103
1,0.0,1.0,30,25.0,1.0,0.0,1.0,0.0,14.876615,14.876615,14.876615,0.0,0.0,0.0,controlled-simulation-from-measured-stage-times,x86_64-40d79c94c103
2,0.0,1.0,30,50.0,1.0,0.0,1.0,0.0,14.876615,14.876615,14.876615,0.0,0.0,0.0,controlled-simulation-from-measured-stage-times,x86_64-40d79c94c103
3,0.0,1.0,60,10.0,1.0,0.0,1.0,0.0,14.876615,14.876615,14.876615,0.0,0.0,0.0,controlled-simulation-from-measured-stage-times,x86_64-40d79c94c103
4,0.0,1.0,60,25.0,1.0,0.0,1.0,0.0,14.876615,14.876615,14.876615,0.0,0.0,0.0,controlled-simulation-from-measured-stage-times,x86_64-40d79c94c103


## 13. Measured concurrent handovers and admission-gating microbenchmarks

The concurrent benchmark drives one shared RSU reference implementation with a serialized state-machine lock and durable SQLite commits. It therefore measures this implementation's realized throughput/queueing behavior rather than reporting `1000/service_time` as if it were observed throughput.

Admission-gating benchmarks separately measure random-PID rejection, known-PID/bad-α0 rejection, and bad-α2 rejection, while asserting that invalid traffic performs zero KEM reservations/decapsulations beyond the one legitimate offer used to create the α2 test.


In [ ]:
def run_concurrency_once(seed:int,concurrency:int,sessions:int)->Dict[str,Any]:
    B=1<<math.ceil(math.log2(max(sessions,1)))
    with tempfile.TemporaryDirectory() as td:
        store=DurableStateStore(Path(td)/"conc.sqlite",B); r=RSU(b"RSU-CONC",BACKEND,epoch=seed,store=store); r.prepare_pool(B)
        jobs=[]
        for i in range(sessions):
            v=Vehicle(hashlib.sha3_256(f"conc-{seed}-{i}".encode()).digest()); w=seed*1_000_000+i; provision(v,r,w); v.cache_pool_root(r.RID,r.epoch,r.root,B,r.sig_pk,r.root_sig,BACKEND); jobs.append((v,w))
        def task(v,w):
            t0=time.perf_counter_ns(); ctx=v.start(r.RID,w); s,m1=r.handle_m0(ctx.m0); m2=v.process_m1(ctx,m1,BACKEND); s2,m3,kr=r.handle_m2(m2); kv=v.process_m3(ctx,m3); t1=time.perf_counter_ns()
            if s!="OFFERED" or s2!="COMPLETED" or not hmac.compare_digest(kr,kv): raise AssertionError("concurrent handover failure")
            r.retire_session(m3.sid); return (t1-t0)/1e6
        t0=time.perf_counter()
        with concurrent.futures.ThreadPoolExecutor(max_workers=min(concurrency,sessions)) as ex:
            lats=list(ex.map(lambda vw: task(*vw),jobs))
        elapsed=time.perf_counter()-t0; store.close()
        return {"seed":seed,"concurrency":concurrency,"sessions":sessions,"elapsed_s":elapsed,"throughput_handover_s":sessions/elapsed,
                "mean_latency_ms":float(np.mean(lats)),"median_latency_ms":float(np.median(lats)),"p95_latency_ms":float(np.quantile(lats,.95)),"p99_latency_ms":float(np.quantile(lats,.99)),
                "evidence_class":EVIDENCE_NATIVE,"platform_label":PLATFORM_LABEL}

concurrency_rows=[]
for seed in SEEDS:
    for c in CONCURRENCY_LEVELS:
        concurrency_rows.append(run_concurrency_once(seed,c,CONCURRENCY_SESSIONS))
concurrency_raw=pd.DataFrame(concurrency_rows)
concurrency_summary=(concurrency_raw.groupby("concurrency",as_index=False)
                     .agg(throughput_mean=("throughput_handover_s","mean"),throughput_sd=("throughput_handover_s","std"),mean_latency_ms=("mean_latency_ms","mean"),p95_latency_ms=("p95_latency_ms","mean"),p99_latency_ms=("p99_latency_ms","mean")))
concurrency_summary["platform_label"]=PLATFORM_LABEL; concurrency_summary["evidence_class"]=EVIDENCE_NATIVE


def flood_benchmark(seed:int,attempts:int)->List[Dict[str,Any]]:
    r=RSU(b"RSU-FLOD",BACKEND,epoch=seed); r.prepare_pool(2); v=Vehicle(hashlib.sha3_256(f"flood-{seed}".encode()).digest()); w=seed; provision(v,r,w); v.cache_pool_root(r.RID,r.epoch,r.root,2,r.sig_pk,r.root_sig,BACKEND)
    ctx=v.start(r.RID,w)
    known_bad=dataclasses.replace(ctx.m0,alpha0=bytes([ctx.m0.alpha0[0]^1])+ctx.m0.alpha0[1:])
    random_bad=M0(hashlib.sha3_256(b"unknown"+u64(seed)).digest()[:16],w,secrets.token_bytes(16),secrets.token_bytes(32))
    rows=[]
    for kind,msg in [("random_PID_M0",random_bad),("known_PID_bad_alpha0",known_bad)]:
        t=[]; reserved0=sum(x.state=="reserved" for x in r.pool); d0=r.decap_calls
        for _ in range(attempts):
            t0=time.perf_counter_ns(); r.handle_m0(msg); t.append((time.perf_counter_ns()-t0)/1e6)
        reserved1=sum(x.state=="reserved" for x in r.pool)
        rows.append({"seed":seed,"attack":kind,"attempts":attempts,"ops_per_s":attempts/(sum(t)/1000),"mean_us":np.mean(t)*1000,"p95_us":np.quantile(t,.95)*1000,
                     "invalid_KEM_reservations":reserved1-reserved0,"invalid_decapsulations":r.decap_calls-d0,"platform_label":PLATFORM_LABEL,"evidence_class":EVIDENCE_NATIVE})
    # Create one legitimate offer then hammer invalid alpha2; it must remain offered and never decapsulate.
    st,m1=r.handle_m0(ctx.m0); good=v.process_m1(ctx,m1,BACKEND); bad=dataclasses.replace(good,alpha2=bytes([good.alpha2[0]^1])+good.alpha2[1:]); d0=r.decap_calls
    t=[]
    for _ in range(attempts):
        t0=time.perf_counter_ns(); s,_,_=r.handle_m2(bad); t.append((time.perf_counter_ns()-t0)/1e6)
        if s!="REJECT_ALPHA2": raise AssertionError(s)
    rows.append({"seed":seed,"attack":"bad_alpha2","attempts":attempts,"ops_per_s":attempts/(sum(t)/1000),"mean_us":np.mean(t)*1000,"p95_us":np.quantile(t,.95)*1000,
                 "invalid_KEM_reservations":0,"invalid_decapsulations":r.decap_calls-d0,"platform_label":PLATFORM_LABEL,"evidence_class":EVIDENCE_NATIVE})
    return rows

flood_raw=pd.DataFrame([row for seed in SEEDS for row in flood_benchmark(seed,FLOOD_ATTEMPTS)])
flood_summary=(flood_raw.groupby("attack",as_index=False).agg(ops_per_s_mean=("ops_per_s","mean"),ops_per_s_sd=("ops_per_s","std"),mean_us=("mean_us","mean"),p95_us=("p95_us","mean"),
                                                            invalid_KEM_reservations=("invalid_KEM_reservations","sum"),invalid_decapsulations=("invalid_decapsulations","sum")))
flood_summary["platform_label"]=PLATFORM_LABEL; flood_summary["evidence_class"]=EVIDENCE_NATIVE
concurrency_summary, flood_summary


(   concurrency  throughput_mean  throughput_sd  mean_latency_ms  \
 0            1       110.547098       8.373210         9.038265   
 1            8       105.214517       5.229717        75.021187   
 2           32       103.161500       7.288052       291.189100   
 3           64       107.450322       7.522130       517.060649   
 4          128       109.355668       5.205083       857.427649   
 
    p95_latency_ms  p99_latency_ms       platform_label   evidence_class  
 0       14.279302       17.092474  x86_64-40d79c94c103  native-measured  
 1      124.267204      153.374536  x86_64-40d79c94c103  native-measured  
 2      458.055625      504.262525  x86_64-40d79c94c103  native-measured  
 3      725.301062      824.028556  x86_64-40d79c94c103  native-measured  
 4     1082.428772     1124.139088  x86_64-40d79c94c103  native-measured  ,
                  attack  ops_per_s_mean   ops_per_s_sd    mean_us     p95_us  \
 0            bad_alpha2    1.829934e+04    2494.791987  5

## 14. Candidate-RSU prepositioning and verifier-window storage

Candidate-specific MIAM values are independently derived. The notebook measures candidate-set derivation cost and provides a transparent steady-state retention model for verifier-window duration. The model assumes `PREPOSITION_ARRIVAL_RATE_HZ` target records issued per second per candidate and reports the assumption explicitly.


In [ ]:
preposition_rows=[]
for seed in SEEDS:
    S=hashlib.sha3_256(f"preposition-{seed}".encode()).digest(); w=seed; c=0
    for C in CANDIDATE_RSU_COUNTS:
        RIDs=[f"R{j:07d}".encode()[:8].ljust(8,b"0") for j in range(C)]
        t0=time.perf_counter_ns(); vals=[derive_miam(S,rid,w,c) for rid in RIDs]; ms=(time.perf_counter_ns()-t0)/1e6
        assert len({a for a,p in vals})==C and len({p for a,p in vals})==C
        preposition_rows.append({"seed":seed,"candidate_RSU_count":C,"derivation_total_ms":ms,"derivation_ms_per_candidate":ms/C,"all_target_values_distinct":True,
                                 "platform_label":PLATFORM_LABEL,"evidence_class":EVIDENCE_NATIVE})
preposition_derivation=pd.DataFrame(preposition_rows)

# Record semantic payload: PID(16)+A(32)+w(8)+c(8)+1 logical state byte = 65 B, excluding DB/index overhead.
TARGET_RECORD_SEMANTIC_BYTES=65
retention=[]
for C in CANDIDATE_RSU_COUNTS:
    for W in VERIFIER_WINDOWS_S:
        active=int(math.ceil(PREPOSITION_ARRIVAL_RATE_HZ*C*W))
        retention.append({"candidate_RSU_count":C,"verifier_window_s":W,"assumed_record_issue_rate_per_s_per_candidate":PREPOSITION_ARRIVAL_RATE_HZ,
                          "modeled_active_records":active,"semantic_storage_bytes":active*TARGET_RECORD_SEMANTIC_BYTES,"semantic_storage_MiB":active*TARGET_RECORD_SEMANTIC_BYTES/2**20,
                          "evidence_class":"explicit-retention-model"})
preposition_storage_model=pd.DataFrame(retention)
preposition_derivation.head(),preposition_storage_model.head()


(   seed  candidate_RSU_count  derivation_total_ms  \
 0  2601                    1              0.29914   
 1  2601                    2              0.18816   
 2  2601                    4              0.28556   
 3  2601                    8              0.61722   
 4  2602                    1              0.39858   
 
    derivation_ms_per_candidate  all_target_values_distinct  \
 0                     0.299140                        True   
 1                     0.094080                        True   
 2                     0.071390                        True   
 3                     0.077152                        True   
 4                     0.398580                        True   
 
         platform_label   evidence_class  
 0  x86_64-40d79c94c103  native-measured  
 1  x86_64-40d79c94c103  native-measured  
 2  x86_64-40d79c94c103  native-measured  
 3  x86_64-40d79c94c103  native-measured  
 4  x86_64-40d79c94c103  native-measured  ,
    candidate_RSU_count  verifier_w

## 15. One-hop healing and compromise-timing boundary checks

These experiments verify that the implementation state exposed by a *past* RSU is insufficient to authorize the next target and map the manuscript's freshness boundary for target/vehicle/MSC compromise. They do **not** empirically prove computational indistinguishability or transcript unlinkability.


In [ ]:
healing_rows=[]
for seed in SEEDS:
    S=hashlib.sha3_256(f"healing-{seed}".encode()).digest(); old=b"OLD-RSU1"; new=b"NEW-RSU1"; w=77; c=5
    Aold,Pold=derive_miam(S,old,w,c); Anew,Pnew=derive_miam(S,new,w,c)
    N0=hashlib.sha3_256(f"n0-{seed}".encode()).digest()[:16]
    oldtag=kmac256(Aold,H(LABELS["ADMIT_CTX"]+enc(old,w,Pold,N0)),32,LABELS["ADMIT"])
    expected_new=kmac256(Anew,H(LABELS["ADMIT_CTX"]+enc(new,w,Pold,N0)),32,LABELS["ADMIT"])
    healing_rows += [
        {"seed":seed,"scenario":"past_RSU_compromised_before_healing","OPMH_applicable":True,"AKE_fresh_possible":True,"healing_distance":1,
         "old_material_authorizes_new_target":hmac.compare_digest(oldtag,expected_new),"new_target_A_in_old_state":False,"new_target_esk_in_old_state":False,
         "modeled_result":"heals at first successful uncompromised target handover","evidence_class":"implementation-boundary-check"},
        {"seed":seed,"scenario":"target_RSU_compromised_before_or_during_decapsulation","OPMH_applicable":False,"AKE_fresh_possible":False,"healing_distance":np.nan,
         "old_material_authorizes_new_target":np.nan,"new_target_A_in_old_state":np.nan,"new_target_esk_in_old_state":np.nan,"modeled_result":"target session non-fresh","evidence_class":"freshness-model-mapping"},
        {"seed":seed,"scenario":"target_RSU_compromised_after_completion_while_Kapp_resident","OPMH_applicable":False,"AKE_fresh_possible":False,"healing_distance":np.nan,
         "old_material_authorizes_new_target":np.nan,"new_target_A_in_old_state":np.nan,"new_target_esk_in_old_state":np.nan,"modeled_result":"resident Kapp treated as revealed","evidence_class":"freshness-model-mapping"},
        {"seed":seed,"scenario":"target_RSU_compromised_after_esk_and_retired_Kapp_erasure","OPMH_applicable":True,"AKE_fresh_possible":True,"healing_distance":1,
         "old_material_authorizes_new_target":False,"new_target_A_in_old_state":False,"new_target_esk_in_old_state":False,"modeled_result":"retired-session secrecy boundary satisfied","evidence_class":"freshness-model-mapping"},
        {"seed":seed,"scenario":"vehicle_root_compromised","OPMH_applicable":False,"AKE_fresh_possible":False,"healing_distance":np.nan,
         "old_material_authorizes_new_target":np.nan,"new_target_A_in_old_state":np.nan,"new_target_esk_in_old_state":np.nan,"modeled_result":"OPMH not claimed","evidence_class":"freshness-model-mapping"},
        {"seed":seed,"scenario":"MSC_database_compromised","OPMH_applicable":False,"AKE_fresh_possible":False,"healing_distance":np.nan,
         "old_material_authorizes_new_target":np.nan,"new_target_A_in_old_state":np.nan,"new_target_esk_in_old_state":np.nan,"modeled_result":"OPMH not claimed","evidence_class":"freshness-model-mapping"},
    ]
healing_boundary=pd.DataFrame(healing_rows)
assert not healing_boundary.loc[healing_boundary.scenario=="past_RSU_compromised_before_healing","old_material_authorizes_new_target"].astype(bool).any()
healing_boundary.head(10)


,seed,scenario,OPMH_applicable,AKE_fresh_possible,healing_distance,old_material_authorizes_new_target,new_target_A_in_old_state,new_target_esk_in_old_state,modeled_result,evidence_class
0,2601,past_RSU_compromised_before_healing,True,True,1.0,False,False,False,heals at first successful uncompromised target...,implementation-boundary-check
1,2601,target_RSU_compromised_before_or_during_decaps...,False,False,NaN,NaN,NaN,NaN,target session non-fresh,freshness-model-mapping
2,2601,target_RSU_compromised_after_completion_while_...,False,False,NaN,NaN,NaN,NaN,resident Kapp treated as revealed,freshness-model-mapping
3,2601,target_RSU_compromised_after_esk_and_retired_K...,True,True,1.0,False,False,False,retired-session secrecy boundary satisfied,freshness-model-mapping
4,2601,vehicle_root_compromised,False,False,NaN,NaN,NaN,NaN,OPMH not claimed,freshness-model-mapping
5,2601,MSC_database_compromised,False,False,NaN,NaN,NaN,NaN,OPMH not claimed,freshness-model-mapping
6,2602,past_RSU_compromised_before_healing,True,True,1.0,False,False,False,heals at first successful uncompromised target...,implementation-boundary-check
7,2602,target_RSU_compromised_before_or_during_decaps...,False,False,NaN,NaN,NaN,NaN,target session non-fresh,freshness-model-mapping
8,2602,target_RSU_compromised_after_completion_while_...,False,False,NaN,NaN,NaN,NaN,resident Kapp treated as revealed,freshness-model-mapping
9,2602,target_RSU_compromised_after_esk_and_retired_K...,True,True,1.0,False,False,False,retired-session secrecy boundary satisfied,freshness-model-mapping


## 16. Functional crash/restart recovery test

This test simulates a process crash **after M1 has been durably committed** but before M2 is processed. A fresh RSU object reconstructs the active offer from SQLite and processes the vehicle's exact M2. A second restart after completion confirms that cached M3 is recoverable while the active `esk` is gone.


In [ ]:
def crash_recovery_test(seed:int)->Dict[str,Any]:
    with tempfile.TemporaryDirectory() as td:
        db=Path(td)/"recover.sqlite"; B=4; store=DurableStateStore(db,B); sigpair=BACKEND.sig_keygen(); r=RSU(b"RSU-CRSH",BACKEND,epoch=seed,store=store,sig_pair=sigpair); r.prepare_pool(B)
        v=Vehicle(hashlib.sha3_256(f"crash-{seed}".encode()).digest()); w=seed; provision(v,r,w); v.cache_pool_root(r.RID,r.epoch,r.root,B,r.sig_pk,r.root_sig,BACKEND)
        ctx=v.start(r.RID,w); _,m1=r.handle_m0(ctx.m0); m2=v.process_m1(ctx,m1,BACKEND); snap=store.snapshot(ctx.PID,w)
        offer_ok=snap["state"]=="offered" and snap["kem_state"]=="reserved" and snap["m0"] is not None and snap["m1"] is not None and snap["esk"] is not None
        store.close()
        # Simulated restart: load only recovery-critical state for the active offer.
        store2=DurableStateStore(db,B); snap=store2.snapshot(ctx.PID,w); rr=RSU(b"RSU-CRSH",BACKEND,epoch=seed,store=store2,sig_pair=sigpair)
        rr.records[(ctx.PID,w)]=TargetRecord(ctx.PID,snap["A"],w,int(snap["c"]),state="offered",accepted_m0_hash=snap["m0h"],j=int(snap["j"]),cached_m0=parse_m0_wire(snap["m0"]),cached_m1=parse_m1_wire(snap["m1"],B))
        rr.pool=[KEMEntry(j+1,b"",None,"spent") for j in range(B)]; j=int(snap["j"]); rr.pool[j-1]=KEMEntry(j,rr.records[(ctx.PID,w)].cached_m1.epk,snap["esk"],"reserved")
        st,m3,kr=rr.handle_m2(m2); kv=v.process_m3(ctx,m3); completion_ok=st=="COMPLETED" and hmac.compare_digest(kr,kv)
        snap2=store2.snapshot(ctx.PID,w); store2.close()
        store3=DurableStateStore(db,B); snap3=store3.snapshot(ctx.PID,w); restart_done_ok=snap3["state"]=="completed" and snap3["kem_state"]=="spent" and snap3["esk"] is None and snap3["m3"]==m3.wire(); store3.close()
        return {"seed":seed,"crash_after_M1_recovery_ok":offer_ok and completion_ok,"post_completion_restart_cached_M3_and_no_esk":restart_done_ok,
                "exact_M0_persisted_for_TH2_reconstruction":snap["m0"] is not None,"evidence_class":"reference-functional-crash-test"}

crash_recovery=pd.DataFrame([crash_recovery_test(s) for s in SEEDS])
assert crash_recovery.drop(columns=["seed","evidence_class"]).all().all()
crash_recovery


,seed,crash_after_M1_recovery_ok,post_completion_restart_cached_M3_and_no_esk,exact_M0_persisted_for_TH2_reconstruction,evidence_class
0,2601,True,True,True,reference-functional-crash-test
1,2602,True,True,True,reference-functional-crash-test
2,2603,True,True,True,reference-functional-crash-test
3,2604,True,True,True,reference-functional-crash-test
4,2605,True,True,True,reference-functional-crash-test
5,2606,True,True,True,reference-functional-crash-test
6,2607,True,True,True,reference-functional-crash-test
7,2608,True,True,True,reference-functional-crash-test
8,2609,True,True,True,reference-functional-crash-test
9,2610,True,True,True,reference-functional-crash-test


## 17. Serialized radio transcript artifact

A synthetic test session is exported as message lengths, SHA-256 digests, and hex-encoded radio messages. No vehicle root secret, MIAM authenticator A, KEM secret key, shared secret Z, PRK, or application key is exported.


In [ ]:
def generate_public_trace()->pd.DataFrame:
    r=RSU(b"RSU-TRCE",BACKEND,epoch=1); r.prepare_pool(8); v=Vehicle(hashlib.sha3_256(b"public-trace-test-root").digest()); w=1; provision(v,r,w); v.cache_pool_root(r.RID,r.epoch,r.root,8,r.sig_pk,r.root_sig,BACKEND)
    ctx=v.start(r.RID,w); _,m1=r.handle_m0(ctx.m0); m2=v.process_m1(ctx,m1,BACKEND); _,m3,kr=r.handle_m2(m2); kv=v.process_m3(ctx,m3); assert hmac.compare_digest(kr,kv)
    rows=[]
    for name,msg in [("M0",ctx.m0),("M1",m1),("M2",m2),("M3",m3)]:
        raw=msg.wire(); rows.append({"message":name,"bytes":len(raw),"sha256":hashlib.sha256(raw).hexdigest(),"wire_hex":raw.hex()})
    return pd.DataFrame(rows)
public_trace=generate_public_trace(); public_trace


,message,bytes,sha256,wire_hex
0,M0,72,1caa8fe761e01e2efeac8df935df7855cec89194fcd7e4...,c19a417a3a5bfed10e058b3b5452020300000000000000...
1,M1,1376,35c2df10e7d43d089afc49bc89c6ab6da18d90070f6803...,5253552d54524345000000000000000100000000000000...
2,M2,1184,bd308c7fc6817504efbc9549c11088a2288eb7d6e33a9a...,c19a417a3a5bfed10e058b3b54520203d2b2f417314472...
3,M3,80,8cb1f6de36b3c51cf58245d61813353d37ee9671b3546d...,5fe4e07140bccb1521ef4fe589af8ef9bd5c7438c97c45...


## 18. Common-platform baselines and optional Tamarin hook

The paper's comparison table may only include a baseline performance number after that baseline is run on the same documented platform/configuration. This notebook therefore exports a schema, not fabricated values. If `mobiheal.spthy` and `tamarin-prover` are present, the notebook runs Tamarin and records the actual exit status/output; otherwise it reports `NOT_RUN`.


In [ ]:
baseline_common_platform=pd.DataFrame(columns=["platform_label","scheme","implementation_commit","PQ_key_establishment","online_signature","V_side_time_ms","RSU_side_time_ms","OTA_bytes","handover_latency_ms","loss_recovery","provenance"])
security_dimensions=pd.DataFrame([
    ("MobiHeal","Yes","Yes*","Yes*","Yes*","Yes","Yes","* under stated freshness/erasure/target-record/vehicle+MSC assumptions"),
    ("LSHSC [5]","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","do not infer"),
    ("Cross-domain [6]","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","do not infer"),
    ("Enhanced auth. [7]","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","do not infer"),
    ("DAME-IoV [11]","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","do not infer"),
    ("PQAKA [12]","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","TO_SOURCE_VERIFY","do not infer"),
],columns=["Scheme","PQ","FS","Past_RSU_containment","Post_compromise_unlinkability","Replay","Loss_recovery","Note"])

TAMARIN_MODEL=Path(os.environ.get("MOBIHEAL_TAMARIN_MODEL","mobiheal.spthy"))
tamarin_status={"status":"NOT_RUN","model":str(TAMARIN_MODEL),"returncode":np.nan,"stdout_file":"","reason":"model or tamarin-prover unavailable"}
if TAMARIN_MODEL.exists() and shutil.which("tamarin-prover"):
    p=subprocess.run(["tamarin-prover",str(TAMARIN_MODEL),"--prove"],capture_output=True,text=True)
    out=RESULTS_DIR/"tamarin_output.txt"; out.write_text(p.stdout+"\nSTDERR\n"+p.stderr,encoding="utf-8")
    tamarin_status={"status":"RUN","model":str(TAMARIN_MODEL),"returncode":p.returncode,"stdout_file":str(out.name),"reason":""}
tamarin_result=pd.DataFrame([tamarin_status])
tamarin_result


,status,model,returncode,stdout_file,reason
0,NOT_RUN,mobiheal.spthy,NaN,,model or tamarin-prover unavailable


## 19. Paper-facing result tables

The MobiHeal row is filled only from measured/accounted outputs of this run. Baseline rows remain `NOT_MEASURED` until common-platform implementations are supplied.


In [ ]:
def metric_mean(name): return float(handover_summary.loc[handover_summary.metric==name,"mean"].iloc[0])
def metric_ci(name):
    x=handover_summary.loc[handover_summary.metric==name].iloc[0]; return float(x.seed_mean_ci95_low),float(x.seed_mean_ci95_high)

m=semantic_sizes(DEFAULT_POOL_B)
paper_table_iii=pd.DataFrame([
    {"Scheme":"MobiHeal","PQ key establishment":"ML-KEM-768","Online signature":"Amortized root only",
     "V-side time ms":metric_mean("vehicle_total_ms"),"V-side 95% CI":str(metric_ci("vehicle_total_ms")),
     "RSU-side time ms":metric_mean("rsu_total_ms"),"RSU-side 95% CI":str(metric_ci("rsu_total_ms")),
     "OTA bytes":m["TOTAL_bytes"],"Handover latency":"see network_summary.csv","Loss recovery":"Yes","Provenance":EVIDENCE_NATIVE+" / accounting"},
    {"Scheme":"DAME-IoV [11]","PQ key establishment":"Lattice/PQ","Online signature":"NOT_SOURCE_VERIFIED","V-side time ms":np.nan,"V-side 95% CI":"","RSU-side time ms":np.nan,"RSU-side 95% CI":"","OTA bytes":np.nan,"Handover latency":"NOT_MEASURED","Loss recovery":"NOT_SOURCE_VERIFIED","Provenance":"NOT_MEASURED"},
    {"Scheme":"PQAKA [12]","PQ key establishment":"ML-KEM","Online signature":"NOT_SOURCE_VERIFIED","V-side time ms":np.nan,"V-side 95% CI":"","RSU-side time ms":np.nan,"RSU-side 95% CI":"","OTA bytes":np.nan,"Handover latency":"NOT_MEASURED","Loss recovery":"NOT_SOURCE_VERIFIED","Provenance":"NOT_MEASURED"},
    {"Scheme":"LQSCPPA [13]","PQ key establishment":"Lattice/PQ","Online signature":"NOT_SOURCE_VERIFIED","V-side time ms":np.nan,"V-side 95% CI":"","RSU-side time ms":np.nan,"RSU-side 95% CI":"","OTA bytes":np.nan,"Handover latency":"NOT_MEASURED","Loss recovery":"NOT_SOURCE_VERIFIED","Provenance":"NOT_MEASURED"},
])

paper_key_results=pd.DataFrame([
    ("online_vehicle_time_ms",metric_mean("vehicle_total_ms"),*metric_ci("vehicle_total_ms"),"native measured, durable-store-independent vehicle path"),
    ("online_RSU_time_ms",metric_mean("rsu_total_ms"),*metric_ci("rsu_total_ms"),"native measured; includes SQLite FULL commits"),
    ("online_local_total_ms",metric_mean("online_local_total_ms"),*metric_ci("online_local_total_ms"),"local compute+durability only, no radio delay"),
    ("OTA_bytes_B1024",m["TOTAL_bytes"],np.nan,np.nan,"exact semantic accounting"),
    ("healing_distance",1,np.nan,np.nan,"modeled OPMH boundary under manuscript freshness assumptions"),
],columns=["metric","value","ci95_low","ci95_high","interpretation"])
paper_table_iii, paper_key_results


(          Scheme PQ key establishment     Online signature  V-side time ms  \
 0       MobiHeal           ML-KEM-768  Amortized root only        0.909613   
 1  DAME-IoV [11]           Lattice/PQ  NOT_SOURCE_VERIFIED             NaN   
 2     PQAKA [12]               ML-KEM  NOT_SOURCE_VERIFIED             NaN   
 3   LQSCPPA [13]           Lattice/PQ  NOT_SOURCE_VERIFIED             NaN   
 
                           V-side 95% CI  RSU-side time ms  \
 0  (0.8936125724500003, 0.924130238325)          6.072514   
 1                                                     NaN   
 2                                                     NaN   
 3                                                     NaN   
 
                       RSU-side 95% CI  OTA bytes         Handover latency  \
 0  (5.76509858545, 6.377599369574999)     3048.0  see network_summary.csv   
 1                                            NaN             NOT_MEASURED   
 2                                            NaN        

## 20. Figures generated only from exported data


In [ ]:
# Primitive timing
fig,ax=plt.subplots(figsize=(9,4.8)); d=primitive_summary.copy(); labels=[f"{a}\n{o}" for a,o in zip(d.algorithm,d.operation)]; ax.bar(labels,d["mean"]); ax.set_ylabel("Mean time (ms)"); ax.set_title(f"MobiHeal primitive timing — {PLATFORM_LABEL}"); ax.tick_params(axis="x",rotation=35); fig.tight_layout(); fig.savefig(FIG_DIR/"primitive_timing.png",dpi=220); plt.close(fig)
# Loss recovery at 10 ms, 90 km/h, 25 m zone
fig,ax=plt.subplots(figsize=(7,4.5)); d=network_summary[(network_summary.one_way_delay_ms==10)&(network_summary.speed_kmh==90)&(network_summary.handover_zone_m==25)]; ax.plot(d.loss*100,d.success_rate*100,marker="o"); ax.set_xlabel("Independent per-message loss (%)"); ax.set_ylabel("Handover success (%)"); ax.set_title("MobiHeal controlled loss-recovery simulation"); ax.set_ylim(0,101); fig.tight_layout(); fig.savefig(FIG_DIR/"loss_recovery.png",dpi=220); plt.close(fig)
# Concurrency
fig,ax=plt.subplots(figsize=(7,4.5)); ax.plot(concurrency_summary.concurrency,concurrency_summary.throughput_mean,marker="o"); ax.set_xlabel("Concurrent client workers"); ax.set_ylabel("Measured handovers/s"); ax.set_title(f"Reference RSU throughput — {PLATFORM_LABEL}"); fig.tight_layout(); fig.savefig(FIG_DIR/"concurrency_throughput.png",dpi=220); plt.close(fig)
# Pool prep amortization
fig,ax=plt.subplots(figsize=(7,4.5)); d=pool_prep_raw.groupby("B",as_index=False).amortized_pool_ms_per_key.mean(); ax.plot(d.B,d.amortized_pool_ms_per_key,marker="o"); ax.set_xscale("log",base=2); ax.set_xlabel("Pool size B"); ax.set_ylabel("Preparation ms / KEM key"); ax.set_title("Offline ephemeral-pool amortization"); fig.tight_layout(); fig.savefig(FIG_DIR/"pool_amortization.png",dpi=220); plt.close(fig)
print("Figures:",FIG_DIR)


Figures: /content/results/figures


## 21. Publication-readiness checks and manuscript-alignment notes

`READY` means the specific experiment was genuinely executed with the required provenance. It does not turn controlled simulation into a wireless testbed or replace the missing Tamarin/baseline artifacts.


In [ ]:
notes=pd.DataFrame([
    {"issue":"Crash recovery metadata","paper_text":"offer tuple lists H(M0), j, NR, M1, toffer","implementation":"also persists exact M0","reason":"TH2 contains enc(M0); exact retransmitted M2 after an RSU crash cannot be authenticated from H(M0) alone","recommended_manuscript_change":"state that exact M0 (or at minimum N0 sufficient to reconstruct M0) is included in the bounded offer recovery record"},
    {"issue":"MSC/vehicle counter synchronization","paper_text":"MSC and vehicle derive current target record using hidden counter c","implementation":"experiment harness provisions using synchronized current counter","reason":"the manuscript does not define the post-handover control-plane synchronization/slow-path procedure","recommended_manuscript_change":"describe the MSC counter-update/resynchronization mechanism or state it as an explicit deployment assumption"},
    {"issue":"Persistent ephemeral-key erasure","paper_text":"persistent active esk is deleted before M3 and erased for abandoned offers","implementation":"SQLite/WAL NULLs the esk for crash-consistency measurement","reason":"ordinary WAL/filesystem/SSD deletion is not forensic secure erasure and may retain stale pages","recommended_manuscript_change":"state that theorem-level deployments require secure erase or crypto-erasure for persistent esk; describe SQLite only as the reference durability benchmark"},
])
checks=pd.DataFrame([
    ("Standards-native ML-KEM-768/ML-DSA-65",BACKEND.standards_native,EVIDENCE_NATIVE,"READY" if BACKEND.standards_native else "BLOCKED"),
    ("Optimized KMAC256",_KMAC256_ACCEL is not None,"native-measured" if _KMAC256_ACCEL else "DEV_ONLY","READY" if _KMAC256_ACCEL else "BLOCKED"),
    ("Protocol/state-machine self-tests",protocol_selftests.passed.all(),"functional","READY"),
    ("Direct online handover timing with durable commits",len(handover_raw)>0,EVIDENCE_NATIVE,"READY" if BACKEND.standards_native else "BLOCKED"),
    ("Offline pool preparation measured",len(pool_prep_raw)>0,EVIDENCE_NATIVE,"READY" if BACKEND.standards_native else "BLOCKED"),
    ("M0-M3 byte accounting",semantic_sizes(1024)["TOTAL_bytes"]==3048,"accounting","READY"),
    ("Controlled loss/delay/speed/dwell simulation",len(network_seed)>0,"controlled-simulation","READY AS SIMULATION"),
    ("Measured concurrency throughput",len(concurrency_raw)>0,EVIDENCE_NATIVE,"READY" if BACKEND.standards_native else "BLOCKED"),
    ("Measured invalid-admission gating cost",len(flood_raw)>0,EVIDENCE_NATIVE,"READY" if BACKEND.standards_native else "BLOCKED"),
    ("Functional crash/restart recovery",crash_recovery.drop(columns=["seed","evidence_class"]).all().all(),"functional","READY"),
    ("Physical secure erasure of persistent esk",False,"PLATFORM_DEPENDENT","REQUIRES secure-delete/crypto-erasure storage on theorem-claiming deployment"),
    ("Common-platform baseline implementations",len(baseline_common_platform)>0,"NOT_MEASURED","BLOCKED UNTIL BASELINES RUN"),
    ("Tamarin machine-checked model",tamarin_status["status"]=="RUN" and tamarin_status["returncode"]==0,"machine-checked" if tamarin_status["status"]=="RUN" else "NOT_RUN","READY" if tamarin_status["status"]=="RUN" and tamarin_status["returncode"]==0 else "BLOCKED/SEPARATE ARTIFACT"),
],columns=["item","executed_or_passed","provenance","status"])
checks


,item,executed_or_passed,provenance,status
0,Standards-native ML-KEM-768/ML-DSA-65,True,native-measured,READY
1,Optimized KMAC256,True,native-measured,READY
2,Protocol/state-machine self-tests,True,functional,READY
3,Direct online handover timing with durable com...,True,native-measured,READY
4,Offline pool preparation measured,True,native-measured,READY
5,M0-M3 byte accounting,True,accounting,READY
6,Controlled loss/delay/speed/dwell simulation,True,controlled-simulation,READY AS SIMULATION
7,Measured concurrency throughput,True,native-measured,READY
8,Measured invalid-admission gating cost,True,native-measured,READY
9,Functional crash/restart recovery,True,functional,READY


## 22. Export CSVs, figures, metadata, hashes, and `results.zip`

A single run regenerates every table/figure from raw data. The archive contains configuration, raw timings, summaries, public transcript traces, environment metadata, package freeze, alignment notes, and SHA-256 checksums.


In [ ]:
CONFIG=pd.DataFrame([
    ("PAPER_MODE",PAPER_MODE),("PLATFORM_LABEL",PLATFORM_LABEL),("SEEDS",json.dumps(SEEDS)),("DEFAULT_POOL_B",DEFAULT_POOL_B),
    ("HANDOVER_WARMUPS",HANDOVER_WARMUPS),("HANDOVER_TRIALS_PER_SEED",HANDOVER_TRIALS_PER_SEED),("PRIMITIVE_REPS",json.dumps(PRIMITIVE_REPS)),
    ("POOL_BENCH_SIZES",json.dumps(POOL_BENCH_SIZES)),("NETWORK_SESSIONS_PER_CELL",NETWORK_SESSIONS_PER_CELL),("NETWORK_LOSS_RATES",json.dumps(NETWORK_LOSS_RATES)),
    ("ONE_WAY_DELAYS_MS",json.dumps(ONE_WAY_DELAYS_MS)),("VEHICLE_SPEEDS_KMH",json.dumps(VEHICLE_SPEEDS_KMH)),("HANDOVER_ZONE_M",json.dumps(HANDOVER_ZONE_M)),
    ("LINK_RATE_MBPS",LINK_RATE_MBPS),("MAX_STAGE_ATTEMPTS",MAX_STAGE_ATTEMPTS),("RTO_MS",RTO_MS),("CONCURRENCY_LEVELS",json.dumps(CONCURRENCY_LEVELS)),
    ("CONCURRENCY_SESSIONS",CONCURRENCY_SESSIONS),("FLOOD_ATTEMPTS",FLOOD_ATTEMPTS),("CANDIDATE_RSU_COUNTS",json.dumps(CANDIDATE_RSU_COUNTS)),
    ("VERIFIER_WINDOWS_S",json.dumps(VERIFIER_WINDOWS_S)),("PREPOSITION_ARRIVAL_RATE_HZ",PREPOSITION_ARRIVAL_RATE_HZ),("BOOTSTRAP_REPS",BOOTSTRAP_REPS),
],columns=["parameter","value"])

exports={
    "environment.csv":ENVIRONMENT,"configuration.csv":CONFIG,"protocol_selftests.csv":protocol_selftests,
    "primitive_raw.csv":primitive_raw,"primitive_summary.csv":primitive_summary,
    "handover_raw.csv":handover_raw,"handover_summary.csv":handover_summary,
    "message_sizes.csv":message_sizes,"pool_storage.csv":pool_storage,"memory_storage_accounting.csv":memory_storage_accounting,
    "pool_preparation_raw.csv":pool_prep_raw,"pool_preparation_summary.csv":pool_prep_summary,
    "network_seed_results.csv":network_seed,"network_summary.csv":network_summary,"network_latency_samples.csv":network_latency_samples,
    "concurrency_raw.csv":concurrency_raw,"concurrency_summary.csv":concurrency_summary,
    "admission_flood_raw.csv":flood_raw,"admission_flood_summary.csv":flood_summary,
    "preposition_derivation.csv":preposition_derivation,"preposition_storage_model.csv":preposition_storage_model,
    "healing_boundary.csv":healing_boundary,"crash_recovery.csv":crash_recovery,"public_transcript_trace.csv":public_trace,
    "paper_table_iii.csv":paper_table_iii,"paper_key_results.csv":paper_key_results,"security_dimensions_template.csv":security_dimensions,
    "baseline_common_platform_template.csv":baseline_common_platform,"tamarin_result.csv":tamarin_result,
    "publication_readiness.csv":checks,"manuscript_alignment_notes.csv":notes,
}
for name,df in exports.items(): df.to_csv(RESULTS_DIR/name,index=False)

(RESULTS_DIR/"pip_freeze.txt").write_text(PIP_FREEZE,encoding="utf-8")
(RESULTS_DIR/"requirements-paper.txt").write_text("pqcrypto==1.0.0\npycryptodome==3.23.0\n# numpy/pandas/matplotlib/psutil exact versions are captured in pip_freeze.txt\n",encoding="utf-8")
(RESULTS_DIR/"execution_command.txt").write_text("MOBIHEAL_PAPER_MODE=1 MOBIHEAL_PLATFORM_LABEL=<platform-name> jupyter nbconvert --to notebook --execute MobiHeal_Paper_Final.ipynb --output MobiHeal_Paper_Final.executed.ipynb\n",encoding="utf-8")

README=f"""# MobiHeal final paper results\n\nGenerated UTC: {pd.Timestamp.utcnow().isoformat()}\nPlatform: {PLATFORM_LABEL}\nBackend: {BACKEND.name}\nStandards-native: {BACKEND.standards_native}\nSeeds: {SEEDS}\n\nEvidence classes\n- native-measured: real standardized PQ/KMAC implementation on this host.\n- accounting: deterministic protocol byte/storage accounting.\n- controlled-simulation-from-measured-stage-times: protocol-level network/mobility Monte Carlo, not a wireless testbed.\n- functional / implementation-boundary-check: executable invariants, not formal security proofs.\n\nPublication cautions\n1. Do not mix literature-reported baseline timings with this run in one common-platform performance table.\n2. Treat network_summary.csv as controlled simulation unless replaced by SUMO/ns-3/testbed results.\n3. Tamarin is a separate machine-checked artifact and is not claimed unless tamarin_result.csv reports a successful actual run.\n4. See manuscript_alignment_notes.csv before finalizing the crash-recovery, counter-synchronization, and persistent-erasure text.\n5. SQLite/WAL timings measure durable ordering; they do not prove forensic secure deletion of old esk bytes.\n"""
(RESULTS_DIR/"README.md").write_text(README,encoding="utf-8")

manifest={"paper":"MobiHeal: Post-Quantum Authenticated Handover with One-Hop Recovery from Past-RSU Compromise for the Internet of Vehicles",
          "generated_utc":pd.Timestamp.utcnow().isoformat(),"platform_label":PLATFORM_LABEL,"backend":BACKEND.name,"standards_native":BACKEND.standards_native,
          "paper_mode":PAPER_MODE,"seeds":SEEDS,"result_files":sorted(exports.keys()),"alignment_notes":len(notes)}
(RESULTS_DIR/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")

# Hash every artifact except the checksum table itself.
checksum=[]
for p in sorted(RESULTS_DIR.rglob("*")):
    if p.is_file() and p.name!="sha256sums.csv":
        checksum.append({"path":str(p.relative_to(RESULTS_DIR)),"sha256":hashlib.sha256(p.read_bytes()).hexdigest(),"bytes":p.stat().st_size})
pd.DataFrame(checksum).to_csv(RESULTS_DIR/"sha256sums.csv",index=False)

ZIP_PATH=ROOT/"results.zip"
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS_DIR.rglob("*")):
        if p.is_file(): z.write(p,arcname=str(Path("results")/p.relative_to(RESULTS_DIR)))
print("Wrote",len(exports),"CSV tables and",ZIP_PATH)
checks


Wrote 31 CSV tables and /content/results.zip


,item,executed_or_passed,provenance,status
0,Standards-native ML-KEM-768/ML-DSA-65,True,native-measured,READY
1,Optimized KMAC256,True,native-measured,READY
2,Protocol/state-machine self-tests,True,functional,READY
3,Direct online handover timing with durable com...,True,native-measured,READY
4,Offline pool preparation measured,True,native-measured,READY
5,M0-M3 byte accounting,True,accounting,READY
6,Controlled loss/delay/speed/dwell simulation,True,controlled-simulation,READY AS SIMULATION
7,Measured concurrency throughput,True,native-measured,READY
8,Measured invalid-admission gating cost,True,native-measured,READY
9,Functional crash/restart recovery,True,functional,READY


## 23. What to report in the paper

For the final manuscript, use `paper_key_results.csv`, `handover_summary.csv`, `primitive_summary.csv`, `message_sizes.csv`, `pool_preparation_summary.csv`, `concurrency_summary.csv`, and the explicitly labeled `network_summary.csv`. Report the platform label, CPU, RAM, OS, crypto package/version, seed count, warm-up count, repetitions, and confidence-interval method from the exported metadata.

Run the same notebook on the second target platform with a different `MOBIHEAL_PLATFORM_LABEL`; do not overwrite or pool the two archives until both runs are complete. Common-platform baseline rows should be added only after those implementations are actually executed under the same setup.
